# B3S — Semantic improvement work

## Why
B4 (held-out) showed that neither H1b_v1 nor GPV_R4_v1 beats unguided B0 in the sparsest scenes (held-out S1 MAE: B0 6.03, GPV_R4 7.91, H1b 9.58); dev100 (20 S1 images) had shown the opposite. dev100 is too small to resolve small sparse-scene effects, so improvement work uses a **larger development pool drawn from the FSC-147 train split** (never used; legitimate for a training-free pipeline; class-disjoint from val/test).

## Pre-declared targets (agreed before any B3S result)
A new version is adopted only if, in a **single pre-declared held-out comparison against H1b_v1**:
1. **S1–S2:** not significantly worse than **B0**;
2. **S5–S6:** significantly better than **H1b_v1**;
3. **Overall MAE:** not worse than H1b_v1.

The FSC-147 test split remains untouched. Development decisions are made on the train pool (dev100 reported for continuity only).

## Development pool — `train_dev400`
- Strata use the **same GT-count boundaries as the validation strata** (derived from val's rank-based qcut), so S1 means the same counts everywhere.
- Fixed-seed sample per stratum: **S1 80, S2 80, S3 60, S4 60, S5 60, S6 60 = 400** (sparse strata doubled for power; dense strata 6× dev100's).
- Results reported per stratum and as stratum-reweighted MAE (population weights 0.20 × 4, 0.15, 0.05).

## Steps in this notebook
| Step | Content | GPU |
|---|---|---|
| 0.1 | Strata boundaries from val; sample `train_dev400` | No |
| 0.2–0.5 | Install models; cache every pipeline input for the pool (resumable) | Yes |
| 1 | Baseline all frozen systems on the pool; check whether the held-out S1 pattern reproduces | No |

# Step 0 — Build and cache the train development pool

In [1]:
# 0.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 0.1 — Paths, repo, strata boundaries, sample train_dev400
# ============================================================

import importlib, json, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
FSC147_RAW = PROJECT_ROOT / "datasets" / "FSC147" / "raw"
FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
B3S_DIR = RESULTS_ROOT / "B3S_Semantic_Improvement"
B3S_DIR.mkdir(parents=True, exist_ok=True)
TD_ROOT = PROJECT_ROOT / "cache" / "FSC147_train_dev400"
TD_ROOT.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.datasets as datasets_mod
import src.pipeline.io_cache as io_cache
for m in (datasets_mod, io_cache):
    importlib.reload(m)

STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]
POOL_SIZES = {"S1_0_20": 80, "S2_20_40": 80, "S3_40_60": 60,
              "S4_60_80": 60, "S5_80_95": 60, "S6_95_100": 60}

def split_meta(split):
    ds = datasets_mod.FSC147Dataset(FSC147_RAW, split=split, load_images=False)
    return pd.DataFrame([{"dataset_index": i, "image_id": ds.image_ids[i],
                          "category": ds.image_to_class[ds.image_ids[i]],
                          "gt_count": len(ds.annotations[ds.image_ids[i]]["points"])}
                         for i in range(len(ds))])

# ---- Val strata exactly as the ABC notebook, then their GT-count boundaries
val_meta = split_meta("val")
val_meta["density_stratum"] = pd.qcut(val_meta["gt_count"].rank(method="first"),
                                      q=[0, .2, .4, .6, .8, .95, 1.0], labels=STRATA).astype(str)
bounds = val_meta.groupby("density_stratum")["gt_count"].agg(["min", "max"]).reindex(STRATA)
print("Validation strata — GT count range:")
display(bounds)

# Upper cut for stratum k = max GT of val stratum k (k = 1..5); counts tied at a
# boundary go to the lower stratum (deterministic, GT-count based)
cuts = [-np.inf] + [float(bounds.loc[s, "max"]) for s in STRATA[:-1]] + [np.inf]

train_meta = split_meta("train")
train_meta["density_stratum"] = pd.cut(train_meta["gt_count"], bins=cuts, labels=STRATA,
                                       right=True).astype(str)
avail = train_meta["density_stratum"].value_counts().reindex(STRATA)
print("\nTrain images available per stratum:")
display(avail.to_frame("available").assign(requested=pd.Series(POOL_SIZES)))
assert all(avail[s] >= POOL_SIZES[s] for s in STRATA), "Not enough train images in a stratum"

rows = []
for j, s in enumerate(STRATA):
    sub = train_meta[train_meta["density_stratum"] == s].sort_values("dataset_index")
    rows.append(sub.sample(n=POOL_SIZES[s], random_state=2026 + j))
td_df = pd.concat(rows).sort_values("dataset_index").reset_index(drop=True)
td_df.to_csv(TD_ROOT / "FSC147_train_dev400.csv", index=False)

print(f"\ntrain_dev400: {len(td_df)} images | categories: {td_df['category'].nunique()}")
display(td_df.groupby("density_stratum")["gt_count"].agg(["size", "min", "median", "max"]).reindex(STRATA))
assert len(set(td_df["image_id"]) & set(val_meta["image_id"])) == 0, "Overlap with val!"

Validation strata — GT count range:


,min,max
density_stratum,,
S1_0_20,7,12
S2_20_40,12,20
S3_40_60,20,37
S4_60_80,37,76
S5_80_95,77,215
S6_95_100,216,2092



Train images available per stratum:


,available,requested
density_stratum,,
S1_0_20,812,80
S2_20_40,700,80
S3_40_60,822,60
S4_60_80,657,60
S5_80_95,571,60
S6_95_100,97,60



train_dev400: 400 images | categories: 66


,size,min,median,max
density_stratum,,,,
S1_0_20,80,8,10.0,12
S2_20_40,80,13,16.0,20
S3_40_60,60,21,27.0,37
S4_60_80,60,38,50.0,75
S5_80_95,60,78,108.0,205
S6_95_100,60,219,288.5,1912


In [3]:
# ============================================================
# Step 0.2 — Install SAN + detectron2, SAM2, Marigold (GPU, T4)
# ============================================================

import importlib.metadata as md_meta
import torch
assert torch.cuda.is_available(), "Switch the runtime to T4."
import src.pipeline.b0 as b0
importlib.reload(b0)

DETECTRON2_COMMIT = "1e3e13bbf607b54f62205c4c33922521822fb298"
FROZEN_SAM2_COMMIT = b0.FROZEN_SAM2_COMMIT

%cd /content
!rm -rf /content/SAN /content/sam2_repo
!git clone -q https://github.com/MendelXu/SAN.git /content/SAN
!pip install -q cython scipy shapely timm h5py submitit scikit-image wandb setuptools fvcore tabulate regex opencv-python "open_clip_torch==2.16.0"
!pip uninstall -y -q detectron2
!python -m pip install -q "git+https://github.com/facebookresearch/detectron2.git@{DETECTRON2_COMMIT}"
!git clone -q https://github.com/facebookresearch/sam2.git /content/sam2_repo
!git -C /content/sam2_repo checkout -q {FROZEN_SAM2_COMMIT}
%cd /content/sam2_repo
!pip install -q -e .
%cd /content
!pip install -q "accelerate==1.14.0" "diffusers==0.40.0" "transformers==5.16.1"
!pip install -q "protobuf==3.20.3" "iopath==0.1.10"   # ABC versions (B3R erratum)

r = subprocess.run([sys.executable, "-c",
    "import sys; sys.path[:0]=['/content/SAN','/content/sam2_repo']; "
    "import google.protobuf, wandb, open_clip, detectron2, san; "
    "from sam2.build_sam import build_sam2; from diffusers import MarigoldDepthPipeline; "
    "print('ALL IMPORTS OK', google.protobuf.__version__)"],
    cwd="/content", capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-2000:])
assert r.returncode == 0
print("If 0.3 reports a protobuf mismatch: restart the session, re-run 0.0-0.1, then 0.3.")

/content
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 12.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver doe

In [4]:
# ============================================================
# Step 0.3 — Build SAN, SAM2 AMG, SAM2 predictor, Marigold
# ============================================================

import shutil
import google.protobuf
assert md_meta.version("protobuf") == "3.20.3" and google.protobuf.__version__ == "3.20.3", \
    "protobuf mismatch — restart, re-run 0.0-0.1, then 0.3."
for p in ["/content/sam2_repo", "/content/SAN"]:
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()

import src.pipeline.semantic as semantic
import src.pipeline.depth as depth_mod
import src.pipeline.prompts as prompts
for m in (b0, semantic, depth_mod, prompts, io_cache):
    importlib.reload(m)

DEVICE = "cuda"
SAN_ROOT = Path("/content/SAN")
SAN_CKPT_LOCAL = SAN_ROOT / "output" / "san_vit_b_16.pth"
SAN_CKPT_LOCAL.parent.mkdir(parents=True, exist_ok=True)
if not SAN_CKPT_LOCAL.exists():
    shutil.copy2(PROJECT_ROOT / "Checkpoints" / "SAN" / "san_vit_b_16.pth", SAN_CKPT_LOCAL)
san_model = semantic.build_san(SAN_ROOT, SAN_CKPT_LOCAL, device=DEVICE)

SAM2_CHECKPOINT = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_small.pt"
amg = b0.build_amg(SAM2_CHECKPOINT, device=DEVICE)                 # frozen B0 setup
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
predictor = SAM2ImagePredictor(build_sam2(b0.FROZEN_MODEL_CFG, str(SAM2_CHECKPOINT), device=DEVICE).eval())
marigold = depth_mod.build_marigold(device=DEVICE)

ENV = {"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "cuda": torch.version.cuda,
       **{k: md_meta.version(k) for k in ["protobuf", "iopath", "detectron2", "diffusers"]}}
print(ENV)


def guided_masks_for_prompts(image_np, prompts_xy):
    h, w = image_np.shape[:2]
    masks, scores = [], []
    if len(prompts_xy):
        with torch.inference_mode():
            predictor.set_image(image_np)
            for p in prompts_xy:
                m, s, _ = predictor.predict(point_coords=np.asarray([p], dtype=np.float32),
                                            point_labels=np.asarray([1], dtype=np.int32),
                                            multimask_output=False)
                masks.append(np.asarray(m[0], dtype=bool)); scores.append(float(np.asarray(s)[0]))
    return (np.asarray(masks) if masks else np.zeros((0, h, w), bool)), np.asarray(scores, float)

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
100%|████████████████████████████████████████| 351M/351M [00:02<00:00, 122MiB/s]
/content/SAN/san/model/side_adapter/timm_wrapper.py:24: UserWarning: Unused kwargs are provided:{'dynamic_img_pad': False, 'device': None, 'dtype': None}.
  warnings.warn(f"Unused kwargs are provided:{kwargs}.")


model_index.json:   0%|          | 0.00/534 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

{'gpu': 'Tesla T4', 'torch': '2.11.0+cu130', 'cuda': '13.0', 'protobuf': '3.20.3', 'iopath': '0.1.10', 'detectron2': '0.6', 'diffusers': '0.40.0'}


In [5]:
# ============================================================
# Step 0.4 — Cache every pipeline input for train_dev400 (resumable)
# ============================================================
#
# Per image: SAN maps -> A3c; B0 AMG masks; Marigold depth; SAN D6
# intermediates -> dense_recovery prompts -> guided SAM2 masks.
# Same formats as the val caches. A done marker is written last.
# ============================================================

from PIL import Image
TD_DIRS = {"b0_masks": TD_ROOT / "b0_masks", "a3c_masks": TD_ROOT / "a3c_masks",
           "depth": TD_ROOT / "depth", "m1_split_labels": TD_ROOT / "m1_split_labels"}
TD_SAN, TD_D6, TD_GUIDED, TD_DONE = (TD_ROOT / "san", TD_ROOT / "d6_intermediate",
                                     TD_ROOT / "guided_dense_recovery", TD_ROOT / "_done")
for d in [TD_DIRS["b0_masks"], TD_DIRS["a3c_masks"], TD_DIRS["depth"], TD_SAN, TD_D6, TD_GUIDED, TD_DONE]:
    d.mkdir(parents=True, exist_ok=True)

def td_guided_path(di, image_id):
    return TD_GUIDED / f"{int(di):04d}_{io_cache.image_stem(image_id)}.npz"

def td_done_path(di, image_id):
    return TD_DONE / f"{int(di):05d}_{io_cache.image_stem(image_id)}.json"

todo = [r for r in td_df.itertuples(index=False) if not td_done_path(r.dataset_index, r.image_id).exists()]
print(f"Already cached: {len(td_df) - len(todo)} | to process: {len(todo)}")
errors, t0 = [], time.time()
for i, r in enumerate(tqdm(todo, desc="train_dev400 caching")):
    try:
        di, iid = r.dataset_index, r.image_id
        p = io_cache.cache_paths(di, iid, TD_DIRS)
        img = Image.open(FSC147_IMAGE_DIR / iid).convert("RGB")
        img_np = np.array(img, copy=True)

        target, background = semantic.run_san(san_model, img, r.category, DEVICE)
        a3c, thr = semantic.a3c_mask(target)
        io_cache.save_san_scores(TD_SAN / io_cache.san_scores_filename(iid), iid, r.category, target, background)
        io_cache.save_a3c_mask(p["a3c_masks"], a3c, thr, semantic.A3C_PERCENTILE, di, iid, r.category)

        bm, bmeta = b0.run_amg(amg, b0.load_image_rgb_cv2(FSC147_IMAGE_DIR / iid))
        io_cache.save_b0_masks(p["b0_masks"], bm, bmeta)

        io_cache.save_depth(p["depth"], depth_mod.run_marigold(marigold, img, DEVICE))

        mp, tq = semantic.run_san_d6(san_model, img, r.category, DEVICE)
        torch.save({"image_id": iid, "target_category": r.category,
                    "mask_preds": mp, "target_query_prob": tq}, TD_D6 / f"{io_cache.image_stem(iid)}.pt")
        pts, info = prompts.dense_recovery_points(a3c, mp, tq)
        gm, gs = guided_masks_for_prompts(img_np, pts)
        io_cache.save_guided_masks(td_guided_path(di, iid), gm, pts, gs, img_np.shape[:2],
                                   dataset_index=di, image_id=iid)

        with open(td_done_path(di, iid), "w") as f:
            json.dump({"dataset_index": int(di), "image_id": iid, "n_b0": int(bm.shape[0]),
                       "n_prompts": int(len(pts)), **info}, f)
    except Exception as e:
        errors.append({"image_id": r.image_id, "error": repr(e)})
        print("ERROR", r.image_id, repr(e))
    if (i + 1) % 50 == 0:
        torch.cuda.empty_cache()
print(f"Elapsed {(time.time() - t0) / 60:.1f} min | errors {len(errors)}")

Already cached: 400 | to process: 0


train_dev400 caching: 0it [00:00, ?it/s]

Elapsed 0.0 min | errors 0


In [6]:
# ============================================================
# Step 0.5 — Validate the pool cache
# ============================================================

ok, problems = 0, []
for r in td_df.itertuples(index=False):
    if not td_done_path(r.dataset_index, r.image_id).exists():
        problems.append((r.image_id, "not done")); continue
    try:
        p = io_cache.cache_paths(r.dataset_index, r.image_id, TD_DIRS)
        io_cache.load_b0_masks(p["b0_masks"]); io_cache.load_a3c_mask(p["a3c_masks"])
        io_cache.load_depth(p["depth"]); io_cache.load_guided_masks(td_guided_path(r.dataset_index, r.image_id))
        io_cache.load_san_scores(TD_SAN / io_cache.san_scores_filename(r.image_id))
        ok += 1
    except Exception as e:
        problems.append((r.image_id, repr(e)))
with open(TD_ROOT / "train_dev400_cache_metadata.json", "w") as f:
    json.dump({"pool": "FSC-147 train, 400 images (80/80/60/60/60/60)", "valid": ok,
               "problems": len(problems), "environment": ENV, "repo_commit": REPO_COMMIT,
               "strata_rule": "val rank-qcut boundaries applied to train GT counts; ties to lower stratum",
               "sampling_seeds": "2026 + stratum index", "test_set_accessed": False,
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}, f, indent=2)
print(f"Valid: {ok}/400 | problems: {len(problems)}")
print("STEP 0:", "COMPLETE" if ok == 400 else "INCOMPLETE — re-run 0.4 (resumes)")

Valid: 400/400 | problems: 0
STEP 0: COMPLETE


# Step 1 — Baseline the frozen systems on train_dev400

All systems through their frozen functions (as B4): B0, S, SG_v1.1, GPV_R4_v1, GPV_K_v1, H1b_v1 (and H2_v1). Reported per stratum and stratum-reweighted. **Key check:** does the held-out sparse pattern reproduce (B0 better than GPV_R4 / H1b in S1)? Paired bootstrap within S1 and S2 (10,000 resamples, seed 0).

**Run order:** CPU is enough for Step 1 → 0.0, 0.1, then Step 1 (the GPU steps are not needed once the cache is complete).

In [3]:
# ============================================================
# Step 1.1 — All systems on train_dev400 (resumable)
# ============================================================

import src.pipeline.sg as sg
import src.pipeline.gpv as gpv
import src.pipeline.multiplicity as K
import src.pipeline.hybrid as hybrid
for m in (io_cache, sg, gpv, K, hybrid):
    importlib.reload(m)
SG_CFG = sg.SGConfig.from_frozen_json(RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "SG_v1.1_frozen_config.json")
TD_DIRS = {"b0_masks": TD_ROOT / "b0_masks", "a3c_masks": TD_ROOT / "a3c_masks",
           "depth": TD_ROOT / "depth", "m1_split_labels": TD_ROOT / "m1_split_labels"}

def compute_systems(b0_masks, b0_iou, guided, a3c, depth, san_target):
    out = {"B0": int(b0_masks.shape[0])}
    s, _ = sg.compute_sg_image(b0_masks, a3c, depth, SG_CFG)
    out.update({"S": int(s["S_count"]), "SG_v1.1": float(s["SG_count"]), "sg_gate": bool(s["stage2_on"])})
    keep = gpv.nms_masks(guided["masks"], guided["scores"], 0.5)
    r4_m, r4_s, _ = gpv.build_r4(guided["masks"][keep], guided["scores"][keep], b0_masks, b0_iou, 0.5)
    out["GPV_R4_v1"] = int(r4_m.shape[0])
    d1 = gpv.d1_peaks(san_target)
    kc, g1 = K.gpv_k_count(r4_m, r4_s, d1, depth)
    out.update({"GPV_K_v1": float(kc), "gate1": bool(g1)})
    out["H1b_v1"] = float(hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)[0])
    out["H2_v1"] = hybrid.h2_route(out["SG_v1.1"], out["sg_gate"], out["GPV_K_v1"], out["gate1"])
    return out

RES = B3S_DIR / "step1_rows"
RES.mkdir(parents=True, exist_ok=True)
todo = [r for r in td_df.itertuples(index=False) if not (RES / f"{int(r.dataset_index):05d}.json").exists()]
print(f"Done: {len(td_df) - len(todo)} | to process: {len(todo)}")
for r in tqdm(todo, desc="Systems"):
    p = io_cache.cache_paths(r.dataset_index, r.image_id, TD_DIRS)
    b0m, b0meta = io_cache.load_b0_masks(p["b0_masks"])
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    g = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery"
                                   / f"{int(r.dataset_index):04d}_{io_cache.image_stem(r.image_id)}.npz")
    res = compute_systems(b0m, b0meta["predicted_iou"], g, a3c, dep, san["target_score"])
    with open(RES / f"{int(r.dataset_index):05d}.json", "w") as f:
        json.dump({"dataset_index": int(r.dataset_index), **res}, f)

td_res = pd.DataFrame([json.load(open(p)) for p in sorted(RES.glob("*.json"))]).merge(td_df, on="dataset_index")
assert len(td_res) == 400

Done: 400 | to process: 0


Systems: 0it [00:00, ?it/s]

In [8]:
# ============================================================
# Step 1.2 — Metrics per stratum + the sparse-pattern check
# ============================================================

SYSTEMS = ["B0", "S", "SG_v1.1", "GPV_R4_v1", "GPV_K_v1", "H1b_v1", "H2_v1"]
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
gt = td_res["gt_count"].to_numpy(float); st = td_res["density_stratum"].to_numpy()

rows = []
for s in SYSTEMS:
    err = td_res[s].to_numpy(float) - gt; ae = np.abs(err)
    row = {"system": s, "MAE_pool": ae.mean(),
           "MAE_reweighted": sum(w * ae[st == k].mean() for k, w in W.items()),
           "RMSE": np.sqrt((err ** 2).mean()), "NAE": (ae / gt).mean(), "mean_signed_error": err.mean()}
    for k in STRATA:
        row[f"MAE_{k[:2]}"] = ae[st == k].mean()
    rows.append(row)
base_td = pd.DataFrame(rows)
print("train_dev400 baseline (frozen systems):")
display(base_td.round(2))

def paired_ci(a, b, mask, n=10_000, seed=0):
    ea = np.abs(td_res.loc[mask, a].to_numpy(float) - gt[mask])
    eb = np.abs(td_res.loc[mask, b].to_numpy(float) - gt[mask])
    idx = np.random.default_rng(seed).integers(0, len(ea), size=(n, len(ea)))
    d = (ea[idx] - eb[idx]).mean(axis=1)
    return ea.mean() - eb.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

print("\nSparse-pattern check (A − B0; positive = worse than B0):")
chk = []
for k in ["S1_0_20", "S2_20_40"]:
    m = st == k
    for a in ["GPV_R4_v1", "H1b_v1", "S", "SG_v1.1"]:
        d, lo, hi = paired_ci(a, "B0", m)
        chk.append({"stratum": k, "system": a, "MAE − B0": d, "CI_low": lo, "CI_high": hi,
                    "significantly_worse": lo > 0})
chk_df = pd.DataFrame(chk)
display(chk_df.round(2))
print("Held-out reference (S1): B0 6.03 | GPV_R4 7.91 | H1b 9.58 | S 8.52 | SG 9.31")

base_td.to_csv(B3S_DIR / "B3S_step1_baseline_train_dev400.csv", index=False)
chk_df.to_csv(B3S_DIR / "B3S_step1_sparse_check.csv", index=False)
td_res.to_csv(B3S_DIR / "B3S_step1_per_image.csv", index=False)
print("Saved to:", B3S_DIR)

train_dev400 baseline (frozen systems):


,system,MAE_pool,MAE_reweighted,RMSE,NAE,mean_signed_error,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0,67.66,35.67,185.27,0.54,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,S,62.05,32.05,180.36,0.55,-51.47,8.40,8.21,13.80,17.37,44.72,315.65
2,SG_v1.1,62.08,32.32,179.49,0.66,-48.50,12.55,9.44,14.25,16.10,41.40,312.83
3,GPV_R4_v1,58.28,29.94,176.09,0.53,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
4,GPV_K_v1,56.69,32.35,174.79,0.74,-20.68,14.90,11.08,26.87,15.22,36.82,264.38
5,H1b_v1,54.82,28.65,169.01,0.57,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
6,H2_v1,57.02,32.78,174.85,0.82,-18.57,18.24,11.68,27.70,14.70,34.27,263.60



Sparse-pattern check (A − B0; positive = worse than B0):


,stratum,system,MAE − B0,CI_low,CI_high,significantly_worse
0,S1_0_20,GPV_R4_v1,3.06,1.62,4.55,True
1,S1_0_20,H1b_v1,4.90,3.02,6.86,True
2,S1_0_20,S,2.84,1.52,4.20,True
3,S1_0_20,SG_v1.1,6.99,3.32,12.78,True
4,S2_20_40,GPV_R4_v1,-0.04,-1.52,1.56,False
5,S2_20_40,H1b_v1,1.75,-0.34,4.03,False
6,S2_20_40,S,0.48,-0.78,1.80,False
7,S2_20_40,SG_v1.1,1.70,0.05,3.60,True


Held-out reference (S1): B0 6.03 | GPV_R4 7.91 | H1b 9.58 | S 8.52 | SG 9.31
Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3S_Semantic_Improvement


# Step 2 — Diagnose the sparse-scene error (S1, with S2 for contrast)

Step 1 showed every semantic addition is significantly worse than B0 in S1 on train_dev400 (GPV_R4 +3.06, H1b +4.90, S +2.84), while in S2 GPV_R4 matches B0. This step asks **where the extra counts come from**, using GT for diagnosis only:

| Step | Question |
|---|---|
| 2.1 | Classify every kept mask (B0 and GPV_R4): **empty** (no object), **redundant** (only objects owned by a smaller mask — part/whole duplicate), **single**, **multi** (merged) |
| 2.2 | Error direction (over vs under), mask composition per image, and whether H1b's local recovery helps or hurts |
| 2.3 | Can the surplus masks be flagged **without GT** (SAM2 score, size, border, SAN scores)? |
| 2.4 | Can a very sparse image be recognised **without GT** (candidate density signals for a count prior)? |

The answers decide which fix to try first: mask verification (vocabulary), prompt selection (integration), or a sparse-scene count prior.

In [9]:
# ============================================================
# Step 2.1 — Per-mask diagnosis of B0 and GPV_R4 masks (train_dev400)
# ============================================================
#
# GT is used for diagnosis only. For every kept mask:
#   empty      : contains no GT point
#   redundant  : contains GT, but every GT point in it is owned by a smaller
#                kept mask (a part/whole duplicate of objects already covered)
#   single     : owns exactly 1 GT point (each GT is owned by the smallest
#                kept mask containing it)
#   multi      : owns >= 2 GT points (a merged mask)
# Plus GT-free mask features for a "could we have flagged it?" check.
# ============================================================

from sklearn.metrics import roc_auc_score

importlib.reload(gpv)
train_ds = datasets_mod.FSC147Dataset(FSC147_RAW, split="train", load_images=False)

def gt_points(di):
    (_, cat), = train_ds[int(di)]["categories"].items()
    return np.asarray(cat["points"], dtype=np.float64).reshape(-1, 2)

def classify_masks(masks, gt_xy):
    # Each GT point is "owned" by the SMALLEST kept mask containing it.
    n = masks.shape[0]
    if n == 0:
        return np.array([], dtype=object), np.zeros(0, int)
    h, w = masks.shape[1:]
    if len(gt_xy) == 0:
        return np.full(n, "empty", dtype=object), np.zeros(n, int)
    px = np.clip(np.rint(gt_xy[:, 0]).astype(int), 0, w - 1)
    py = np.clip(np.rint(gt_xy[:, 1]).astype(int), 0, h - 1)
    hits = masks[:, py, px]                                   # (N, G)
    areas = masks.reshape(n, -1).sum(axis=1)
    gt_in = hits.sum(axis=1)
    owns = np.zeros(n, int)
    for g_i in range(hits.shape[1]):
        holders = np.flatnonzero(hits[:, g_i])
        if len(holders):
            owns[holders[np.argmin(areas[holders])]] += 1
    cls = np.where(gt_in == 0, "empty",
          np.where(owns == 0, "redundant",
          np.where(owns == 1, "single", "multi"))).astype(object)
    return cls, gt_in

mask_rows = []
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="Mask diagnosis"):
    di = r.dataset_index
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    b0m, b0meta = io_cache.load_b0_masks(p["b0_masks"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    g = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery"
                                   / f"{int(di):04d}_{io_cache.image_stem(r.image_id)}.npz")
    keep = gpv.nms_masks(g["masks"], g["scores"], 0.5)
    r4_m, r4_s, r4_src = gpv.build_r4(g["masks"][keep], g["scores"][keep], b0m,
                                      b0meta["predicted_iou"], 0.5)
    gt_xy = gt_points(di)
    t, b = san["target_score"], san["background_score"]
    img_area = t.size
    for system, masks, scores, src in [("B0", b0m, b0meta["predicted_iou"], np.full(len(b0m), "b0", object)),
                                       ("GPV_R4", r4_m, r4_s, r4_src)]:
        cls, gt_in = classify_masks(masks, gt_xy)
        if masks.shape[0] == 0:
            continue
        ev = gpv.mask_semantic_evidence(masks, t, b)
        areas = masks.reshape(masks.shape[0], -1).sum(axis=1)
        border = masks[:, 0, :].any(1) | masks[:, -1, :].any(1) | masks[:, :, 0].any(1) | masks[:, :, -1].any(1)
        med_area = float(np.median(areas))
        for j in range(masks.shape[0]):
            mask_rows.append({"dataset_index": di, "density_stratum": r.density_stratum,
                              "gt_count": r.gt_count, "system": system, "source": src[j],
                              "class": cls[j], "gt_inside": int(gt_in[j]),
                              "area_frac": areas[j] / img_area, "area_over_median": areas[j] / med_area,
                              "touches_border": bool(border[j]), "sam_score": float(scores[j]),
                              "mean_target": float(ev["mean_target"][j]),
                              "mean_background": float(ev["mean_background"][j]),
                              "target_pixel_fraction": float(ev["target_pixel_fraction"][j])})
masks_df = pd.DataFrame(mask_rows)
print(f"Masks analysed: {len(masks_df)}")

Mask diagnosis:   0%|          | 0/400 [00:00<?, ?it/s]

Masks analysed: 32253


In [10]:
# ============================================================
# Step 2.2 — Where do the S1/S2 errors come from?
# ============================================================

SP = ["S1_0_20", "S2_20_40"]

# ---- (a) Error direction per image
dir_rows = []
for k in SP + ["S3_40_60"]:
    sub = td_res[td_res["density_stratum"] == k]
    for s in ["B0", "S", "GPV_R4_v1", "H1b_v1"]:
        e = sub[s] - sub["gt_count"]
        dir_rows.append({"stratum": k, "system": s, "over": int((e > 0).sum()), "exact": int((e == 0).sum()),
                         "under": int((e < 0).sum()), "mean_signed": e.mean(), "MAE": e.abs().mean()})
print("(a) Error direction per image:")
display(pd.DataFrame(dir_rows).round(2))

# ---- (b) Mask composition per image (B0 vs GPV_R4), mean per image
comp = (masks_df[masks_df["density_stratum"].isin(SP)]
        .groupby(["density_stratum", "system", "source", "class"]).size()
        .unstack("class", fill_value=0))
n_img = td_df.groupby("density_stratum").size()
comp_per_img = comp.div(comp.index.get_level_values("density_stratum").map(n_img).values, axis=0)
print("\n(b) Mean masks per image by class (S1, S2):")
display(comp_per_img.round(2))

surplus = (masks_df[masks_df["density_stratum"].isin(SP)]
           .assign(is_empty=lambda d: d["class"] == "empty",
                   is_redundant=lambda d: d["class"] == "redundant")
           .groupby(["density_stratum", "system"])[["is_empty", "is_redundant"]].sum())
surplus = surplus.div(surplus.index.get_level_values("density_stratum").map(n_img).values, axis=0)
print("Surplus masks per image (empty + redundant):")
display(surplus.round(2))

# ---- (c) H1b vs GPV_R4 in S1/S2: does local recovery help or hurt?
h_rows = []
for k in SP:
    sub = td_res[td_res["density_stratum"] == k]
    diff = sub["H1b_v1"] - sub["GPV_R4_v1"]
    ch = diff != 0
    better = ((sub["H1b_v1"] - sub["gt_count"]).abs() < (sub["GPV_R4_v1"] - sub["gt_count"]).abs())[ch]
    h_rows.append({"stratum": k, "images_changed": int(ch.sum()), "units_added": float(diff[ch].sum()),
                   "changed_and_better": int(better.sum()), "changed_and_worse": int((~better).sum())})
print("\n(c) H1b's local recovery relative to GPV_R4:")
display(pd.DataFrame(h_rows))

(a) Error direction per image:


,stratum,system,over,exact,under,mean_signed,MAE
0,S1_0_20,B0,41,3,36,1.41,5.56
1,S1_0_20,S,66,3,11,7.72,8.40
2,S1_0_20,GPV_R4_v1,74,3,3,8.50,8.62
3,S1_0_20,H1b_v1,76,3,1,10.44,10.46
4,S2_20_40,B0,39,1,40,0.04,7.74
5,S2_20_40,S,57,3,20,5.64,8.21
6,S2_20_40,GPV_R4_v1,64,5,11,5.88,7.70
7,S2_20_40,H1b_v1,67,6,7,8.46,9.49
8,S3_40_60,B0,24,0,36,-6.37,13.30
9,S3_40_60,S,38,1,21,4.87,13.80



(b) Mean masks per image by class (S1, S2):


class                          empty  multi  redundant  single
density_stratum system source                                 
S1_0_20         B0     b0       4.85   0.04       1.08    5.82
                GPV_R4 b0       4.58   0.00       0.30    5.24
                       guided   4.22   0.24       0.50    3.80
S2_20_40        B0     b0       5.08   0.02       1.76    9.55
                GPV_R4 b0       4.60   0.02       0.52    8.96
                       guided   3.09   0.22       0.24    4.59

Surplus masks per image (empty + redundant):


is_empty  is_redundant
density_stratum system                        
S1_0_20         B0          4.85          1.08
                GPV_R4      8.80          0.80
S2_20_40        B0          5.08          1.76
                GPV_R4      7.69          0.76


(c) H1b's local recovery relative to GPV_R4:


,stratum,images_changed,units_added,changed_and_better,changed_and_worse
0,S1_0_20,29,155.0,11,18
1,S2_20_40,30,207.0,13,17


In [11]:
# ============================================================
# Step 2.3 — Could the surplus masks be flagged GT-free?
# ============================================================
#
# AUC of each GT-free mask feature for separating surplus masks
# (empty or redundant) from useful masks (single or multi), in S1-S2,
# separately for guided-prompt masks and B0 masks in GPV_R4.
# AUC 0.5 = no signal; far from 0.5 (either side) = usable signal.
# ============================================================

FEATS = ["sam_score", "area_frac", "area_over_median", "touches_border",
         "mean_target", "mean_background", "target_pixel_fraction"]
sp = masks_df[(masks_df["system"] == "GPV_R4") & masks_df["density_stratum"].isin(SP)].copy()
sp["surplus"] = sp["class"].isin(["empty", "redundant"])
auc_rows = []
for src in ["guided", "b0"]:
    d = sp[sp["source"] == src]
    if d["surplus"].nunique() < 2:
        continue
    for f in FEATS:
        auc_rows.append({"source": src, "feature": f, "n_masks": len(d),
                         "surplus_share": d["surplus"].mean(),
                         "AUC_surplus_vs_useful": roc_auc_score(d["surplus"], d[f].astype(float))})
auc_df = pd.DataFrame(auc_rows)
auc_df["strength"] = (auc_df["AUC_surplus_vs_useful"] - 0.5).abs()
print("GT-free features vs surplus masks (GPV_R4, S1-S2):")
display(auc_df.sort_values(["source", "strength"], ascending=[True, False]).round(3))

print("\nFeature means by class (GPV_R4 guided masks, S1-S2):")
display(sp[sp["source"] == "guided"].groupby("class")[FEATS].mean().round(3))

GT-free features vs surplus masks (GPV_R4, S1-S2):


,source,feature,n_masks,surplus_share,AUC_surplus_vs_useful,strength
11,b0,mean_target,1938,0.413,0.206,0.294
12,b0,mean_background,1938,0.413,0.742,0.242
13,b0,target_pixel_fraction,1938,0.413,0.294,0.206
7,b0,sam_score,1938,0.413,0.315,0.185
10,b0,touches_border,1938,0.413,0.660,0.160
8,b0,area_frac,1938,0.413,0.369,0.131
9,b0,area_over_median,1938,0.413,0.422,0.078
5,guided,mean_background,1352,0.476,0.699,0.199
0,guided,sam_score,1352,0.476,0.306,0.194
3,guided,touches_border,1352,0.476,0.600,0.100



Feature means by class (GPV_R4 guided masks, S1-S2):


,sam_score,area_frac,area_over_median,touches_border,mean_target,mean_background,target_pixel_fraction
class,,,,,,,
empty,0.689,0.036,11.731,0.291,2.257,0.076,0.900
multi,0.755,0.159,32.274,0.270,1.630,0.020,0.966
redundant,0.692,0.136,12.026,0.373,2.875,0.069,0.959
single,0.829,0.019,1.725,0.089,2.101,0.025,0.997


In [12]:
# ============================================================
# Step 2.4 — Candidate GT-free density signals (for a count prior)
# ============================================================
#
# Image-level features available at inference. Spearman correlation
# with GT count, and AUC for "S1 vs the rest" (can we tell a very
# sparse image without GT?).
# ============================================================

from scipy.stats import spearmanr

img_feats = []
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="Image features"):
    di = r.dataset_index
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    b0m, _ = io_cache.load_b0_masks(p["b0_masks"])
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    g = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery"
                                   / f"{int(di):04d}_{io_cache.image_stem(r.image_id)}.npz")
    t, bg = san["target_score"], san["background_score"]
    labels, areas, _, _ = sg.c1_components(a3c)
    b0_areas = b0m.reshape(b0m.shape[0], -1).sum(axis=1) if b0m.shape[0] else np.zeros(0)
    img_feats.append({
        "dataset_index": di,
        "n_b0_masks": int(b0m.shape[0]),
        "n_prompts": int(len(g["prompts_xy"])),
        "n_d1_peaks": int(len(gpv.d1_peaks(t))),
        "n_c1_components": int(areas.size),
        "median_b0_area_frac": float(np.median(b0_areas) / t.size) if b0_areas.size else np.nan,
        "target_gt_bg_fraction": float((t > bg).mean()),
        "target_mean": float(t.mean()),
        "target_p90_over_p50": float(np.percentile(t, 90) / max(np.percentile(t, 50), 1e-6)),
    })
img_df = pd.DataFrame(img_feats).merge(td_df[["dataset_index", "gt_count", "density_stratum"]], on="dataset_index")
is_s1 = img_df["density_stratum"] == "S1_0_20"
sig_rows = []
for f in [c for c in img_df.columns if c not in ("dataset_index", "gt_count", "density_stratum")]:
    x = img_df[f].astype(float)
    ok = x.notna()
    sig_rows.append({"feature": f,
                     "spearman_with_GT": spearmanr(x[ok], img_df.loc[ok, "gt_count"]).correlation,
                     "AUC_S1_vs_rest": roc_auc_score(is_s1[ok], x[ok])})
sig_df = pd.DataFrame(sig_rows)
sig_df["S1_separability"] = (sig_df["AUC_S1_vs_rest"] - 0.5).abs()
print("GT-free image-level density signals:")
display(sig_df.sort_values("S1_separability", ascending=False).round(3))

# ---- Save
D2 = B3S_DIR / "step2_sparse_diagnosis"
D2.mkdir(parents=True, exist_ok=True)
masks_df.to_csv(D2 / "B3S_s2_masks.csv", index=False)
auc_df.to_csv(D2 / "B3S_s2_mask_feature_auc.csv", index=False)
img_df.to_csv(D2 / "B3S_s2_image_features.csv", index=False)
sig_df.to_csv(D2 / "B3S_s2_density_signals.csv", index=False)
print("Saved to:", D2)

Image features:   0%|          | 0/400 [00:00<?, ?it/s]

GT-free image-level density signals:


,feature,spearman_with_GT,AUC_S1_vs_rest,S1_separability
3,n_c1_components,0.509,0.255,0.245
1,n_prompts,0.499,0.278,0.222
0,n_b0_masks,0.426,0.287,0.213
2,n_d1_peaks,0.469,0.302,0.198
7,target_p90_over_p50,-0.281,0.661,0.161
5,target_gt_bg_fraction,0.388,0.346,0.154
4,median_b0_area_frac,-0.320,0.626,0.126
6,target_mean,0.185,0.398,0.102


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3S_Semantic_Improvement/step2_sparse_diagnosis


# Step 3 — Expanded SAN vocabulary (diagnostic)

Step 2 showed the S1 error is almost entirely **empty masks** — large, low-confidence, border-touching background regions — about half from B0 and half from guided prompts. Existing SAN evidence separates them only moderately (AUC ≈ 0.70–0.80): with "background" as the only alternative, ~90% of an empty mask's pixels still prefer the target.

This step re-runs SAN with **[target, "background", + 43 pre-declared COCO-Stuff background classes]** (object-like stuff classes excluded; terms overlapping the class name removed) and measures, without changing any pipeline:

| Step | Question |
|---|---|
| 3.1 | SAN maps with the expanded vocabulary (GPU) |
| 3.2 | Do new mask features separate empty/redundant from useful masks better than the old ones? |
| 3.3 | What would a parameter-free rule — *drop a mask whose dominant class is not the target* — do per stratum? (diagnostic, not adopted) |
| 3.4 | Is the image-level target share a better density signal than Step 2.4's? |

In [13]:
# ============================================================
# Step 3.1 — SAN with the expanded vocabulary on train_dev400 (GPU, resumable)
# ============================================================
#
# Requires Step 0.3 (san_model built). Vocabulary per image:
#   [class name, "background", + pre-declared background stuff classes]
#   (stuff terms overlapping the class name are removed).
# Saved per image (compact): per-pixel argmax class index, target score,
# best non-target score, and the vocabulary used.
# ============================================================

import src.pipeline.semantic as semantic
importlib.reload(semantic)
assert hasattr(semantic, "run_san_vocab"), "Old semantic.py — commit the Step 3 version."
assert "san_model" in globals(), "Run Steps 0.2-0.3 first (GPU)."

TD_VOCAB = TD_ROOT / "san_vocab_v1"
TD_VOCAB.mkdir(parents=True, exist_ok=True)
from PIL import Image

todo = [r for r in td_df.itertuples(index=False)
        if not (TD_VOCAB / f"{io_cache.image_stem(r.image_id)}.npz").exists()]
print(f"Done: {len(td_df) - len(todo)} | to process: {len(todo)} | "
      f"stuff classes: {len(semantic.STUFF_BACKGROUND_VOCAB)}")
t0 = time.time()
for r in tqdm(todo, desc="SAN expanded vocabulary"):
    img = Image.open(FSC147_IMAGE_DIR / r.image_id).convert("RGB")
    vocab = semantic.expanded_vocabulary(r.category)
    seg = semantic.run_san_vocab(san_model, img, vocab, DEVICE)        # (C, H, W)
    best_other = seg[1:].max(axis=0)
    np.savez_compressed(TD_VOCAB / f"{io_cache.image_stem(r.image_id)}.npz",
                        argmax=seg.argmax(axis=0).astype(np.uint8),
                        target=seg[0].astype(np.float16),
                        best_other=best_other.astype(np.float16),
                        vocabulary=np.array(vocab))
print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")

Done: 400 | to process: 0 | stuff classes: 43


SAN expanded vocabulary: 0it [00:00, ?it/s]

Elapsed: 0.0 min


In [14]:
# ============================================================
# Step 3.2 — New mask evidence vs the old (AUC, S1-S2)
# ============================================================
#
# New GT-free mask features from the expanded vocabulary:
#   target_win_fraction : share of mask pixels whose argmax class is the target
#   dominant_is_target  : the most frequent argmax class inside the mask is the target
#   mean_margin         : mean (target score - best non-target score) inside the mask
# Compared with Step 2 features on the same masks (same order).
# ============================================================

assert "masks_df" in globals(), "Run Step 2.1 first."
masks_df["mask_order"] = masks_df.groupby(["dataset_index", "system"]).cumcount()
TD_VOCAB = TD_ROOT / "san_vocab_v1"

new_rows = []
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="New mask evidence"):
    di = r.dataset_index
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    b0m, b0meta = io_cache.load_b0_masks(p["b0_masks"])
    g = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery"
                                   / f"{int(di):04d}_{io_cache.image_stem(r.image_id)}.npz")
    keep = gpv.nms_masks(g["masks"], g["scores"], 0.5)
    r4_m, _, _ = gpv.build_r4(g["masks"][keep], g["scores"][keep], b0m, b0meta["predicted_iou"], 0.5)
    with np.load(TD_VOCAB / f"{io_cache.image_stem(r.image_id)}.npz") as v:
        amax = v["argmax"]; margin = v["target"].astype(np.float32) - v["best_other"].astype(np.float32)
        n_cls = len(v["vocabulary"])
    win = (amax == 0)
    for system, masks in [("B0", b0m), ("GPV_R4", r4_m)]:
        for j in range(masks.shape[0]):
            m = masks[j]
            if not m.any():
                tw, dom, mm = 0.0, -1, 0.0
            else:
                tw = float(win[m].mean())
                dom = int(np.bincount(amax[m], minlength=n_cls).argmax())
                mm = float(margin[m].mean())
            new_rows.append({"dataset_index": di, "system": system, "mask_order": j,
                             "target_win_fraction": tw, "dominant_class": dom,
                             "dominant_is_target": dom == 0, "mean_margin": mm})
new_df = pd.DataFrame(new_rows)
mf = masks_df.merge(new_df, on=["dataset_index", "system", "mask_order"], validate="one_to_one")
assert len(mf) == len(masks_df), "Mask order mismatch with Step 2.1"

SP = ["S1_0_20", "S2_20_40"]
sp = mf[(mf["system"] == "GPV_R4") & mf["density_stratum"].isin(SP)].copy()
sp["surplus"] = sp["class"].isin(["empty", "redundant"])
OLD = ["sam_score", "area_over_median", "touches_border", "mean_target", "mean_background",
       "target_pixel_fraction"]
NEW = ["target_win_fraction", "dominant_is_target", "mean_margin"]
rows = []
for src in ["guided", "b0"]:
    d = sp[sp["source"] == src]
    for f in OLD + NEW:
        auc = roc_auc_score(d["surplus"], d[f].astype(float))
        rows.append({"source": src, "feature": f, "family": "new (expanded vocab)" if f in NEW else "old",
                     "AUC": auc, "strength": abs(auc - 0.5)})
auc3 = pd.DataFrame(rows).sort_values(["source", "strength"], ascending=[True, False])
print("Surplus (empty/redundant) vs useful masks — GPV_R4, S1-S2:")
display(auc3.round(3))

print("\nNew features by mask class (GPV_R4, S1-S2):")
display(sp.groupby(["source", "class"])[NEW].mean().round(3))

New mask evidence:   0%|          | 0/400 [00:00<?, ?it/s]

Surplus (empty/redundant) vs useful masks — GPV_R4, S1-S2:


,source,feature,family,AUC,strength
17,b0,mean_margin,new (expanded vocab),0.197,0.303
12,b0,mean_target,old,0.206,0.294
16,b0,dominant_is_target,new (expanded vocab),0.219,0.281
15,b0,target_win_fraction,new (expanded vocab),0.245,0.255
13,b0,mean_background,old,0.742,0.242
14,b0,target_pixel_fraction,old,0.294,0.206
9,b0,sam_score,old,0.315,0.185
11,b0,touches_border,old,0.660,0.160
10,b0,area_over_median,old,0.422,0.078
4,guided,mean_background,old,0.699,0.199



New features by mask class (GPV_R4, S1-S2):


target_win_fraction  dominant_is_target  mean_margin
source class                                                          
b0     empty                    0.381               0.377        0.326
       multi                    0.999               1.000        3.412
       redundant                0.644               0.636        1.803
       single                   0.939               0.960        2.570
guided empty                    0.749               0.756        1.703
       multi                    0.906               0.973        1.326
       redundant                0.914               0.949        2.600
       single                   0.906               0.952        1.822

In [15]:
# ============================================================
# Step 3.3 — Diagnostic: parameter-free rule "drop masks whose dominant class is not the target"
# ============================================================
#
# NOT adopted here — measures what such a rule would do, per stratum:
#   masks dropped, share of dropped that are surplus (empty/redundant),
#   useful masks lost, and the count MAE before/after, for B0 and GPV_R4.
# ============================================================

gt_by_img = td_df.set_index("dataset_index")["gt_count"]
eff_rows = []
for system in ["B0", "GPV_R4"]:
    d = mf[mf["system"] == system]
    for k in STRATA:
        dk = d[d["density_stratum"] == k]
        imgs = td_df.loc[td_df["density_stratum"] == k, "dataset_index"]
        before = dk.groupby("dataset_index").size().reindex(imgs, fill_value=0)
        after = dk[dk["dominant_is_target"]].groupby("dataset_index").size().reindex(imgs, fill_value=0)
        gt_k = gt_by_img.loc[imgs]
        dropped = dk[~dk["dominant_is_target"]]
        eff_rows.append({
            "system": system, "stratum": k,
            "masks_per_img": before.mean(), "dropped_per_img": len(dropped) / len(imgs),
            "dropped_that_are_surplus": dropped["class"].isin(["empty", "redundant"]).mean() if len(dropped) else np.nan,
            "useful_lost_per_img": dropped["class"].isin(["single", "multi"]).sum() / len(imgs),
            "MAE_before": np.abs(before.values - gt_k.values).mean(),
            "MAE_after": np.abs(after.values - gt_k.values).mean(),
        })
eff_df = pd.DataFrame(eff_rows)
display(eff_df.round(2))
print("Targets for reference — S1/S2 B0: 5.56 / 7.74 ; S5/S6 H1b: 37.33 / 274.98")

,system,stratum,masks_per_img,dropped_per_img,dropped_that_are_surplus,useful_lost_per_img,MAE_before,MAE_after
0,B0,S1_0_20,11.79,3.10,0.99,0.02,5.56,5.39
1,B0,S2_20_40,16.41,4.00,0.85,0.61,7.74,8.59
2,B0,S3_40_60,21.00,2.65,0.82,0.48,13.30,14.95
3,B0,S4_60_80,33.13,6.13,0.57,2.65,22.93,28.50
4,B0,S5_80_95,54.93,6.83,0.29,4.87,59.03,65.83
5,B0,S6_95_100,69.17,8.73,0.27,6.33,338.08,346.82
6,GPV_R4,S1_0_20,18.88,3.96,0.98,0.06,8.62,5.64
7,GPV_R4,S2_20_40,22.25,4.85,0.81,0.91,7.70,6.40
8,GPV_R4,S3_40_60,33.18,4.30,0.70,1.28,12.45,12.92
9,GPV_R4,S4_60_80,47.23,10.38,0.59,4.30,16.13,19.92


Targets for reference — S1/S2 B0: 5.56 / 7.74 ; S5/S6 H1b: 37.33 / 274.98


In [16]:
# ============================================================
# Step 3.4 — Image-level target share as a density signal; save
# ============================================================

dens = []
for r in td_df.itertuples(index=False):
    with np.load(TD_VOCAB / f"{io_cache.image_stem(r.image_id)}.npz") as v:
        amax = v["argmax"]
    dens.append({"dataset_index": r.dataset_index, "target_share_vocab": float((amax == 0).mean())})
dens_df = pd.DataFrame(dens).merge(td_df[["dataset_index", "gt_count", "density_stratum"]], on="dataset_index")
is_s1 = dens_df["density_stratum"] == "S1_0_20"
rho = spearmanr(dens_df["target_share_vocab"], dens_df["gt_count"]).correlation
auc_s1 = roc_auc_score(is_s1, dens_df["target_share_vocab"])
print(f"target_share_vocab: Spearman with GT {rho:.3f} | AUC S1-vs-rest {auc_s1:.3f} "
      f"(separability {abs(auc_s1 - 0.5):.3f})")
print("Step 2.4 best old signal: n_c1_components, separability 0.245")
display(dens_df.groupby("density_stratum")["target_share_vocab"].describe().reindex(STRATA).round(3))

D3 = B3S_DIR / "step3_expanded_vocabulary"
D3.mkdir(parents=True, exist_ok=True)
mf.to_csv(D3 / "B3S_s3_mask_features.csv", index=False)
auc3.to_csv(D3 / "B3S_s3_auc_old_vs_new.csv", index=False)
eff_df.to_csv(D3 / "B3S_s3_dominant_rule_effect.csv", index=False)
dens_df.to_csv(D3 / "B3S_s3_target_share.csv", index=False)
with open(D3 / "B3S_s3_metadata.json", "w") as f:
    json.dump({"step": "3 — expanded SAN vocabulary (diagnostic)",
               "vocabulary": "[class, background] + STUFF_BACKGROUND_VOCAB (43 terms; overlaps with class name removed)",
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "rule_adopted": False,
               "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", D3)

target_share_vocab: Spearman with GT 0.265 | AUC S1-vs-rest 0.388 (separability 0.112)
Step 2.4 best old signal: n_c1_components, separability 0.245


,count,mean,std,min,25%,50%,75%,max
density_stratum,,,,,,,,
S1_0_20,80.0,0.395,0.272,0.009,0.167,0.377,0.600,1.0
S2_20_40,80.0,0.423,0.315,0.000,0.097,0.444,0.636,1.0
S3_40_60,60.0,0.531,0.366,0.000,0.127,0.580,0.910,1.0
S4_60_80,60.0,0.511,0.357,0.000,0.202,0.515,0.820,1.0
S5_80_95,60.0,0.561,0.340,0.000,0.334,0.527,0.921,1.0
S6_95_100,60.0,0.665,0.352,0.000,0.401,0.776,0.986,1.0


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3S_Semantic_Improvement/step3_expanded_vocabulary


# Step 4 — Semantic verification (V-a, V-b), judged against the agreed targets

**Pre-declared before running:**
- **V-a:** drop a kept mask if its dominant expanded-vocabulary class is not the target.
- **V-b:** drop a kept mask only if its dominant class is not the target **and** its area > 4 × the median kept-mask area (existing group-mask convention; motivated by Step 2: empty masks ≈ 11.7× median, real objects ≈ 1.7×).
- Applied to **GPV_R4** (count = kept masks) and to **H1b** (verification on R4's masks first, then H1b unchanged). **B0 + V-b** reported as a reference.

**Targets (development check on train_dev400; final judgement remains the single held-out comparison):**
- **T1:** S1 and S2 not significantly worse than B0.
- **T2:** S5 and S6 significantly better than H1b_v1.
- **T3:** stratum-reweighted MAE not significantly worse than H1b_v1.

In [17]:
# ============================================================
# Step 4.1 — Apply V-a / V-b to GPV_R4, H1b (and B0) on train_dev400
# ============================================================

import src.pipeline.hybrid as hybrid
for m in (gpv, hybrid, sg):
    importlib.reload(m)
TD_VOCAB = TD_ROOT / "san_vocab_v1"
GROUP_FACTOR = 4.0


def verify_keep(masks, amax, rule):
    # Boolean keep-vector for V-a / V-b (dominant class from the expanded vocabulary)
    n = masks.shape[0]
    if n == 0:
        return np.zeros(0, bool)
    n_cls = int(amax.max()) + 1
    dominant = np.array([np.bincount(amax[m], minlength=n_cls).argmax() if m.any() else -1
                         for m in masks])
    not_target = dominant != 0
    if rule == "V-a":
        return ~not_target
    areas = masks.reshape(n, -1).sum(axis=1)
    large = areas > GROUP_FACTOR * np.median(areas)
    return ~(not_target & large)


v_rows = []
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="Step 4 variants"):
    di = r.dataset_index
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    b0m, b0meta = io_cache.load_b0_masks(p["b0_masks"])
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    g = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery"
                                   / f"{int(di):04d}_{io_cache.image_stem(r.image_id)}.npz")
    with np.load(TD_VOCAB / f"{io_cache.image_stem(r.image_id)}.npz") as v:
        amax = v["argmax"]
    keep = gpv.nms_masks(g["masks"], g["scores"], 0.5)
    r4_m, r4_s, _ = gpv.build_r4(g["masks"][keep], g["scores"][keep], b0m, b0meta["predicted_iou"], 0.5)
    d1 = gpv.d1_peaks(san["target_score"])

    row = {"dataset_index": di, "GPV_R4": int(r4_m.shape[0])}
    for rule in ["V-a", "V-b"]:
        k = verify_keep(r4_m, amax, rule)
        row[f"GPV_R4+{rule}"] = int(k.sum())
        row[f"H1b+{rule}"] = float(hybrid.h1b_count(r4_m[k], d1, a3c, dep, SG_CFG)[0])
        row[f"dropped_{rule}"] = int((~k).sum())
    row["B0+V-b"] = int(verify_keep(b0m, amax, "V-b").sum())
    v_rows.append(row)

v_df = (pd.DataFrame(v_rows)
        .merge(td_res[["dataset_index", "B0", "H1b_v1", "GPV_R4_v1", "gt_count", "density_stratum"]],
               on="dataset_index", validate="one_to_one"))
assert (v_df["GPV_R4"] == v_df["GPV_R4_v1"]).all(), "R4 rebuild differs from Step 1"
print("Images:", len(v_df))

Step 4 variants:   0%|          | 0/400 [00:00<?, ?it/s]

Images: 400


In [18]:
# ============================================================
# Step 4.2 — Per-stratum results and the agreed targets (paired bootstrap)
# ============================================================
#
# Targets (development check on train_dev400):
#   T1  S1 and S2 : not significantly worse than B0   (CI of [system - B0] has lower bound <= 0)
#   T2  S5 and S6 : significantly better than H1b_v1  (CI of [system - H1b] has upper bound < 0)
#   T3  overall   : stratum-reweighted MAE not significantly worse than H1b_v1
# Bootstrap: 10,000 resamples, within-stratum resampling (seeds 1000 + stratum index).
# ============================================================

SYS4 = ["B0", "B0+V-b", "GPV_R4_v1", "GPV_R4+V-a", "GPV_R4+V-b", "H1b_v1", "H1b+V-a", "H1b+V-b"]
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
N_BOOT = 10_000
gt4 = v_df["gt_count"].to_numpy(float); st4 = v_df["density_stratum"].to_numpy()
AE = {s: np.abs(v_df[s].to_numpy(float) - gt4) for s in SYS4}

# ---- point estimates
rows = []
for s in SYS4:
    row = {"system": s, "MAE_reweighted": sum(w * AE[s][st4 == k].mean() for k, w in W.items()),
           "mean_signed": float((v_df[s] - v_df["gt_count"]).mean())}
    for k in STRATA:
        row[f"MAE_{k[:2]}"] = AE[s][st4 == k].mean()
    rows.append(row)
res4 = pd.DataFrame(rows)
display(res4.round(2))

# ---- within-stratum bootstrap indices (shared by all systems -> paired)
IDX = {}
for j, k in enumerate(STRATA):
    pos = np.flatnonzero(st4 == k)
    IDX[k] = pos[np.random.default_rng(1000 + j).integers(0, len(pos), size=(N_BOOT, len(pos)))]

def strat_ci(a, b, k):
    d = AE[a][IDX[k]].mean(axis=1) - AE[b][IDX[k]].mean(axis=1)
    return AE[a][st4 == k].mean() - AE[b][st4 == k].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

def rw_ci(a, b):
    d = sum(w * (AE[a][IDX[k]].mean(axis=1) - AE[b][IDX[k]].mean(axis=1)) for k, w in W.items())
    obs = sum(w * (AE[a][st4 == k].mean() - AE[b][st4 == k].mean()) for k, w in W.items())
    return obs, np.percentile(d, 2.5), np.percentile(d, 97.5)

t_rows = []
for s in ["GPV_R4_v1", "GPV_R4+V-a", "GPV_R4+V-b", "H1b_v1", "H1b+V-a", "H1b+V-b"]:
    rec = {"system": s}
    t1 = []
    for k in ["S1_0_20", "S2_20_40"]:
        d, lo, hi = strat_ci(s, "B0", k)
        rec[f"{k[:2]} vs B0"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
        t1.append(lo <= 0)
    t2 = []
    for k in ["S5_80_95", "S6_95_100"]:
        if s == "H1b_v1":
            rec[f"{k[:2]} vs H1b"] = "—"; t2.append(False); continue
        d, lo, hi = strat_ci(s, "H1b_v1", k)
        rec[f"{k[:2]} vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
        t2.append(hi < 0)
    if s == "H1b_v1":
        rec["overall vs H1b"] = "—"; t3 = True
    else:
        d, lo, hi = rw_ci(s, "H1b_v1")
        rec["overall vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
        t3 = lo <= 0
    rec.update({"T1 (S1,S2 ok)": all(t1), "T2 (S5,S6 better)": all(t2), "T3 (overall ok)": t3,
                "ALL TARGETS": all(t1) and all(t2) and t3})
    t_rows.append(rec)
targets4 = pd.DataFrame(t_rows)
print("\nTargets (difference, 95% CI; negative = system better):")
display(targets4)

drop_tab = v_df.groupby("density_stratum")[["dropped_V-a", "dropped_V-b"]].mean().reindex(STRATA)
print("\nMasks dropped per image:")
display(drop_tab.round(2))

D4 = B3S_DIR / "step4_verification"
D4.mkdir(parents=True, exist_ok=True)
v_df.to_csv(D4 / "B3S_s4_per_image.csv", index=False)
res4.to_csv(D4 / "B3S_s4_metrics.csv", index=False)
targets4.to_csv(D4 / "B3S_s4_targets.csv", index=False)
with open(D4 / "B3S_s4_metadata.json", "w") as f:
    json.dump({"step": "4 — semantic verification V-a / V-b (train_dev400)",
               "rules": {"V-a": "drop kept mask if dominant expanded-vocabulary class != target",
                         "V-b": "drop only if dominant != target AND area > 4 x median kept-mask area"},
               "applied_to": ["GPV_R4 (count = kept)", "H1b (verification on R4 masks, then H1b unchanged)",
                              "B0 (V-b, reference)"],
               "targets": {"T1": "S1,S2 not significantly worse than B0",
                           "T2": "S5,S6 significantly better than H1b_v1",
                           "T3": "reweighted MAE not significantly worse than H1b_v1"},
               "n_boot": N_BOOT, "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", D4)

,system,MAE_reweighted,mean_signed,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0,35.67,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,B0+V-b,35.92,-63.33,5.42,7.84,13.57,23.38,59.42,339.20
2,GPV_R4_v1,29.94,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
3,GPV_R4+V-a,32.14,-54.80,5.64,6.40,12.92,19.92,49.62,314.42
4,GPV_R4+V-b,29.81,-48.01,7.69,7.10,12.32,16.17,41.17,299.62
5,H1b_v1,28.65,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
6,H1b+V-a,30.98,-50.75,6.70,7.62,12.85,18.75,46.38,296.67
7,H1b+V-b,28.79,-43.74,9.30,8.65,12.18,14.90,37.93,281.85



Targets (difference, 95% CI; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,"T1 (S1,S2 ok)","T2 (S5,S6 better)",T3 (overall ok),ALL TARGETS
0,GPV_R4_v1,"+3.06 [+1.64, +4.54]","-0.04 [-1.51, +1.59]","+3.18 [-0.97, +9.07]","+22.65 [+14.58, +31.80]","+1.29 [+0.32, +2.39]",False,False,False,False
1,GPV_R4+V-a,"+0.08 [-1.56, +1.69]","-1.34 [-2.85, +0.26]","+12.28 [+5.87, +20.30]","+39.43 [+27.58, +52.65]","+3.49 [+2.06, +5.04]",True,False,False,False
2,GPV_R4+V-b,"+2.12 [+0.77, +3.56]","-0.64 [-2.11, +0.99]","+3.83 [-0.43, +9.78]","+24.63 [+16.52, +33.75]","+1.16 [+0.16, +2.29]",False,False,False,False
3,H1b_v1,"+4.90 [+3.04, +6.85]","+1.75 [-0.36, +4.10]",—,—,—,False,False,True,False
4,H1b+V-a,"+1.14 [-0.75, +2.95]","-0.11 [-2.15, +2.12]","+9.05 [+4.07, +15.45]","+21.68 [+10.95, +34.47]","+2.33 [+1.15, +3.66]",True,False,False,False
5,H1b+V-b,"+3.74 [+2.03, +5.51]","+0.91 [-1.20, +3.29]","+0.60 [-1.60, +2.33]","+6.87 [+2.58, +13.53]","+0.14 [-0.37, +0.66]",False,False,True,False



Masks dropped per image:


,dropped_V-a,dropped_V-b
density_stratum,,
S1_0_20,3.96,1.06
S2_20_40,4.85,0.98
S3_40_60,4.30,1.10
S4_60_80,10.38,1.07
S5_80_95,10.20,1.32
S6_95_100,16.78,1.98


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3S_Semantic_Improvement/step4_verification


# Step 5 — SAN tiling diagnostic (V-a with sharper maps)

Step 4: V-a fixes S1–S2 (GPV_R4 + V-a: S1 5.64 ≈ B0 5.56; S2 6.40 < B0 7.74) but loses real objects in dense scenes — at SAN's ~1/16 grid, a small object's mask is dominated by its surroundings. V-b's size condition was too blunt (it dropped only ~1 mask per image).

**Pre-declared map variants:**
- **tiled:** expanded-vocabulary SAN on 2 × 2 corner-anchored tiles, each 60% of the image height/width (20% overlap), each through SAN's usual preprocessing; class scores averaged where tiles overlap.
- **tiled + full:** average of the tiled map and the full-image map (keeps whole-scene context for large objects).

**Revised targets (agreed before Step 5 results):**
- **T1:** S1, S2 not significantly worse than B0.
- **T2′:** S5, S6 not significantly worse than H1b_v1.
- **T3′:** stratum-reweighted MAE significantly **better** than H1b_v1.

| Step | Content |
|---|---|
| 5.1 | Tiled SAN maps (GPU) |
| 5.2 | Mechanism check: do real single-object masks in S4–S6 now have the target as dominant class, while empty masks in S1–S2 still do not? |
| 5.3 | V-a with each map on GPV_R4 and H1b, judged against T1, T2′, T3′ |

In [19]:
# ============================================================
# Step 5.1 — Tiled SAN (expanded vocabulary) on train_dev400 (GPU, resumable)
# ============================================================
#
# Pre-declared:
#   tiles   : 2 x 2 overlapping tiles, each 60% of the image height / width,
#             anchored at the four corners (overlap = 20% of each side);
#             each tile goes through SAN's usual preprocessing (short side 640)
#   tiled   : per-class score maps averaged over overlapping tiles
#   tiled+full : average of the tiled map and the full-image map
# Saved per image: argmax / target / best non-target for both variants.
# ============================================================

import src.pipeline.semantic as semantic
importlib.reload(semantic)
assert "san_model" in globals(), "Run Steps 0.2-0.3 first (GPU)."
from PIL import Image

TD_TILED = TD_ROOT / "san_vocab_tiled"
TD_TILED.mkdir(parents=True, exist_ok=True)
TILE_FRAC = 0.60


def tiled_scores(img, vocab):
    w, h = img.size
    th, tw = int(np.ceil(TILE_FRAC * h)), int(np.ceil(TILE_FRAC * w))
    acc = np.zeros((len(vocab), h, w), np.float32)
    cnt = np.zeros((h, w), np.float32)
    for y0 in (0, h - th):
        for x0 in (0, w - tw):
            tile = img.crop((x0, y0, x0 + tw, y0 + th))
            seg = semantic.run_san_vocab(san_model, tile, vocab, DEVICE)      # (C, th, tw)
            acc[:, y0:y0 + th, x0:x0 + tw] += seg
            cnt[y0:y0 + th, x0:x0 + tw] += 1.0
    return acc / cnt[None]


def pack(seg):
    return {"argmax": seg.argmax(axis=0).astype(np.uint8),
            "target": seg[0].astype(np.float16),
            "best_other": seg[1:].max(axis=0).astype(np.float16)}


todo = [r for r in td_df.itertuples(index=False)
        if not (TD_TILED / f"{io_cache.image_stem(r.image_id)}.npz").exists()]
print(f"Done: {len(td_df) - len(todo)} | to process: {len(todo)}")
t0 = time.time()
for r in tqdm(todo, desc="Tiled SAN"):
    img = Image.open(FSC147_IMAGE_DIR / r.image_id).convert("RGB")
    vocab = semantic.expanded_vocabulary(r.category)
    full = semantic.run_san_vocab(san_model, img, vocab, DEVICE)
    tiled = tiled_scores(img, vocab)
    both = 0.5 * (tiled + full)
    pt, pb = pack(tiled), pack(both)
    np.savez_compressed(TD_TILED / f"{io_cache.image_stem(r.image_id)}.npz",
                        tiled_argmax=pt["argmax"], tiled_target=pt["target"], tiled_best_other=pt["best_other"],
                        both_argmax=pb["argmax"], both_target=pb["target"], both_best_other=pb["best_other"],
                        vocabulary=np.array(vocab))
print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")

Done: 0 | to process: 400


Tiled SAN:   0%|          | 0/400 [00:00<?, ?it/s]

Elapsed: 8.3 min


In [20]:
# ============================================================
# Step 5.2 — Mechanism check: dominant class per mask, full vs tiled vs tiled+full
# ============================================================
#
# GT for diagnosis only. For GPV_R4 kept masks:
#   single masks in S4-S6 : share whose dominant class is the target (want HIGH)
#   empty  masks in S1-S2 : share whose dominant class is the target (want LOW)
# ============================================================

from sklearn.metrics import roc_auc_score
TD_VOCAB = TD_ROOT / "san_vocab_v1"
train_ds = datasets_mod.FSC147Dataset(FSC147_RAW, split="train", load_images=False)

def gt_points(di):
    (_, cat), = train_ds[int(di)]["categories"].items()
    return np.asarray(cat["points"], dtype=np.float64).reshape(-1, 2)

def classify_masks(masks, gt_xy):
    n = masks.shape[0]
    if n == 0:
        return np.array([], dtype=object)
    h, w = masks.shape[1:]
    if len(gt_xy) == 0:
        return np.full(n, "empty", dtype=object)
    px = np.clip(np.rint(gt_xy[:, 0]).astype(int), 0, w - 1)
    py = np.clip(np.rint(gt_xy[:, 1]).astype(int), 0, h - 1)
    hits = masks[:, py, px]
    areas = masks.reshape(n, -1).sum(axis=1)
    gt_in = hits.sum(axis=1)
    owns = np.zeros(n, int)
    for g_i in range(hits.shape[1]):
        holders = np.flatnonzero(hits[:, g_i])
        if len(holders):
            owns[holders[np.argmin(areas[holders])]] += 1
    return np.where(gt_in == 0, "empty", np.where(owns == 0, "redundant",
                    np.where(owns == 1, "single", "multi"))).astype(object)

def dominant(masks, amax):
    n_cls = int(amax.max()) + 1
    return np.array([np.bincount(amax[m], minlength=n_cls).argmax() if m.any() else -1 for m in masks])

MAPS = ["full", "tiled", "both"]
R4_CACHE, mech_rows = {}, []
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="Mechanism check"):
    di = r.dataset_index
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    b0m, b0meta = io_cache.load_b0_masks(p["b0_masks"])
    g = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery"
                                   / f"{int(di):04d}_{io_cache.image_stem(r.image_id)}.npz")
    keep = gpv.nms_masks(g["masks"], g["scores"], 0.5)
    r4_m, r4_s, r4_src = gpv.build_r4(g["masks"][keep], g["scores"][keep], b0m, b0meta["predicted_iou"], 0.5)
    stem = io_cache.image_stem(r.image_id)
    with np.load(TD_VOCAB / f"{stem}.npz") as v:
        am = {"full": v["argmax"]}
    with np.load(TD_TILED / f"{stem}.npz") as v:
        am["tiled"], am["both"] = v["tiled_argmax"], v["both_argmax"]
    dom = {k: dominant(r4_m, am[k]) for k in MAPS} if r4_m.shape[0] else {k: np.zeros(0) for k in MAPS}
    cls = classify_masks(r4_m, gt_points(di))
    R4_CACHE[di] = (r4_m, dom)
    for j in range(r4_m.shape[0]):
        mech_rows.append({"dataset_index": di, "density_stratum": r.density_stratum, "source": r4_src[j],
                          "class": cls[j], **{f"dom_target_{k}": bool(dom[k][j] == 0) for k in MAPS}})
mech = pd.DataFrame(mech_rows)

dense = mech[mech["density_stratum"].isin(["S4_60_80", "S5_80_95", "S6_95_100"]) & (mech["class"] == "single")]
sparse = mech[mech["density_stratum"].isin(["S1_0_20", "S2_20_40"]) & (mech["class"] == "empty")]
tab = pd.DataFrame({
    "single masks S4-S6: dominant = target (want HIGH)": [dense[f"dom_target_{k}"].mean() for k in MAPS],
    "empty masks S1-S2: dominant = target (want LOW)": [sparse[f"dom_target_{k}"].mean() for k in MAPS],
}, index=MAPS)
display(tab.round(3))
print(f"(n single masks S4-S6 = {len(dense)}, n empty masks S1-S2 = {len(sparse)})")
print("\nBy source (empty masks S1-S2, dominant = target):")
display(sparse.groupby("source")[[f"dom_target_{k}" for k in MAPS]].mean().round(3))

Mechanism check:   0%|          | 0/400 [00:00<?, ?it/s]

,single masks S4-S6: dominant = target (want HIGH),empty masks S1-S2: dominant = target (want LOW)
full,0.884,0.545
tiled,0.891,0.537
both,0.895,0.550


(n single masks S4-S6 = 11452, n empty masks S1-S2 = 1319)

By source (empty masks S1-S2, dominant = target):


,dom_target_full,dom_target_tiled,dom_target_both
source,,,
b0,0.377,0.392,0.392
guided,0.756,0.718,0.749


In [21]:
# ============================================================
# Step 5.3 — V-a with each map on GPV_R4 and H1b; revised targets
# ============================================================
#
# Revised targets (agreed before Step 5 results):
#   T1  S1, S2 : not significantly worse than B0      (CI lower bound <= 0)
#   T2' S5, S6 : not significantly worse than H1b_v1  (CI lower bound <= 0)
#   T3' overall: stratum-reweighted MAE significantly BETTER than H1b_v1 (CI upper bound < 0)
# ============================================================

import src.pipeline.hybrid as hybrid
importlib.reload(hybrid)

c_rows = []
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="V-a with each map"):
    di = r.dataset_index
    r4_m, dom = R4_CACHE[di]
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    d1 = gpv.d1_peaks(san["target_score"])
    row = {"dataset_index": di}
    for k in MAPS:
        keep = dom[k] == 0 if r4_m.shape[0] else np.zeros(0, bool)
        row[f"GPV_R4+V-a[{k}]"] = int(keep.sum())
        row[f"H1b+V-a[{k}]"] = float(hybrid.h1b_count(r4_m[keep], d1, a3c, dep, SG_CFG)[0])
    c_rows.append(row)
c_df = pd.DataFrame(c_rows).merge(td_res[["dataset_index", "B0", "GPV_R4_v1", "H1b_v1",
                                          "gt_count", "density_stratum"]], on="dataset_index")

SYS5 = ["B0", "GPV_R4_v1", "H1b_v1"] + [f"{a}+V-a[{k}]" for a in ["GPV_R4", "H1b"] for k in MAPS]
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
gt5 = c_df["gt_count"].to_numpy(float); st5 = c_df["density_stratum"].to_numpy()
AE5 = {s: np.abs(c_df[s].to_numpy(float) - gt5) for s in SYS5}
IDX5 = {}
for j, k in enumerate(STRATA):
    pos = np.flatnonzero(st5 == k)
    IDX5[k] = pos[np.random.default_rng(1000 + j).integers(0, len(pos), size=(10_000, len(pos)))]

def sci(a, b, k):
    d = AE5[a][IDX5[k]].mean(1) - AE5[b][IDX5[k]].mean(1)
    return AE5[a][st5 == k].mean() - AE5[b][st5 == k].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

def rci(a, b):
    d = sum(w * (AE5[a][IDX5[k]].mean(1) - AE5[b][IDX5[k]].mean(1)) for k, w in W.items())
    o = sum(w * (AE5[a][st5 == k].mean() - AE5[b][st5 == k].mean()) for k, w in W.items())
    return o, np.percentile(d, 2.5), np.percentile(d, 97.5)

res5 = pd.DataFrame([{"system": s,
                      "MAE_reweighted": sum(w * AE5[s][st5 == k].mean() for k, w in W.items()),
                      **{f"MAE_{k[:2]}": AE5[s][st5 == k].mean() for k in STRATA}} for s in SYS5])
display(res5.round(2))

t_rows = []
for s in SYS5[3:]:
    rec = {"system": s}
    t1 = []
    for k in ["S1_0_20", "S2_20_40"]:
        d, lo, hi = sci(s, "B0", k); rec[f"{k[:2]} vs B0"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t1.append(lo <= 0)
    t2 = []
    for k in ["S5_80_95", "S6_95_100"]:
        d, lo, hi = sci(s, "H1b_v1", k); rec[f"{k[:2]} vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t2.append(lo <= 0)
    d, lo, hi = rci(s, "H1b_v1"); rec["overall vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
    rec.update({"T1": all(t1), "T2'": all(t2), "T3'": hi < 0, "ALL": all(t1) and all(t2) and hi < 0})
    t_rows.append(rec)
targets5 = pd.DataFrame(t_rows)
print("\nRevised targets (difference [95% CI]; negative = system better):")
display(targets5)

D5 = B3S_DIR / "step5_tiling"
D5.mkdir(parents=True, exist_ok=True)
mech.to_csv(D5 / "B3S_s5_mechanism_masks.csv", index=False)
c_df.to_csv(D5 / "B3S_s5_per_image.csv", index=False)
res5.to_csv(D5 / "B3S_s5_metrics.csv", index=False)
targets5.to_csv(D5 / "B3S_s5_targets.csv", index=False)
with open(D5 / "B3S_s5_metadata.json", "w") as f:
    json.dump({"step": "5 — SAN tiling diagnostic with V-a",
               "tiles": f"2x2 corner-anchored, {TILE_FRAC:.0%} of each side (20% overlap); scores averaged",
               "maps": {"full": "Step 3 expanded-vocabulary map", "tiled": "tiled average",
                        "both": "0.5 x (tiled + full)"},
               "targets": {"T1": "S1,S2 not significantly worse than B0",
                           "T2'": "S5,S6 not significantly worse than H1b_v1",
                           "T3'": "reweighted MAE significantly better than H1b_v1"},
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", D5)

V-a with each map:   0%|          | 0/400 [00:00<?, ?it/s]

,system,MAE_reweighted,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0,35.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,10.46,9.49,12.35,14.20,37.33,274.98
3,GPV_R4+V-a[full],32.14,5.64,6.40,12.92,19.92,49.62,314.42
4,GPV_R4+V-a[tiled],31.33,5.70,6.19,12.10,18.13,47.30,316.22
5,GPV_R4+V-a[both],31.42,5.78,6.18,12.33,18.62,47.27,315.08
6,H1b+V-a[full],30.98,6.70,7.62,12.85,18.75,46.38,296.67
7,H1b+V-a[tiled],30.16,6.72,6.91,11.80,16.82,44.50,300.68
8,H1b+V-a[both],30.21,6.76,7.34,12.02,17.40,44.15,297.72



Revised targets (difference [95% CI]; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,GPV_R4+V-a[full],"+0.08 [-1.56, +1.69]","-1.34 [-2.85, +0.26]","+12.28 [+5.87, +20.30]","+39.43 [+27.58, +52.65]","+3.49 [+2.06, +5.04]",True,False,False,False
1,GPV_R4+V-a[tiled],"+0.14 [-1.44, +1.71]","-1.55 [-3.09, +0.08]","+9.97 [+3.75, +17.78]","+41.23 [+28.53, +55.42]","+2.68 [+1.24, +4.22]",True,False,False,False
2,GPV_R4+V-a[both],"+0.21 [-1.39, +1.81]","-1.56 [-3.09, +0.08]","+9.93 [+3.68, +17.72]","+40.10 [+27.92, +53.67]","+2.77 [+1.35, +4.31]",True,False,False,False
3,H1b+V-a[full],"+1.14 [-0.75, +2.95]","-0.11 [-2.15, +2.12]","+9.05 [+4.07, +15.45]","+21.68 [+10.95, +34.47]","+2.33 [+1.15, +3.66]",True,False,False,False
4,H1b+V-a[tiled],"+1.16 [-0.66, +2.95]","-0.83 [-2.61, +1.03]","+7.17 [+2.67, +13.30]","+25.70 [+13.43, +40.05]","+1.51 [+0.34, +2.81]",True,False,False,False
5,H1b+V-a[both],"+1.20 [-0.66, +3.00]","-0.40 [-2.45, +1.84]","+6.82 [+2.28, +13.05]","+22.73 [+11.72, +35.87]","+1.56 [+0.43, +2.82]",True,False,False,False


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3S_Semantic_Improvement/step5_tiling


# Step 6 — Count-gated verification (option A; final test of this line)

Step 5 showed V-a's dense-scene harm is not a resolution problem (88% of real S4–S6 object masks already have the target dominant; tiling adds < 1 pp). It is the **undercount effect**: pipelines over-count in S1–S2 (removing masks helps) and undercount in S4–S6 (removing any mask hurts). Verification should therefore act only where over-counting is likely.

**Pre-declared rule (before any Step 6 result):**
- **Gate:** apply V-a only if the image's GPV_R4 count ≤ **20** — the upper GT bound of validation stratum S2, taken from the strata definition, not fitted.
- **V-a:** drop a kept mask if its dominant expanded-vocabulary class is not the target.
- **Maps:** tiled (primary), full (secondary).
- **Systems:** GPV_R4 + gated V-a; H1b + gated V-a (verification on R4 masks, then H1b unchanged; the gate uses the R4 count before verification).

**Targets:** T1 (S1, S2 not significantly worse than B0), T2′ (S5, S6 not significantly worse than H1b_v1), T3′ (reweighted MAE significantly better than H1b_v1).

**Commitment:** this is the last iteration of the verification line. If a variant passes all three targets it goes to the single held-out comparison against H1b_v1; otherwise verification is reported as a finding and H1b_v1 remains the primary.

In [4]:
# ============================================================
# Step 6.1 — Count-gated verification (CPU, cached inputs)
# ============================================================
#
# Pre-declared (before any Step 6 result):
#   gate    : apply V-a only if the image's GPV_R4 count <= COUNT_GATE,
#             COUNT_GATE = upper GT bound of validation stratum S2 (= 20),
#             taken from the strata definition, not fitted.
#   V-a     : drop a kept mask if its dominant expanded-vocabulary class != target
#   maps    : tiled (primary) and full (secondary)
#   systems : GPV_R4 + gated V-a ; H1b + gated V-a (verification on R4 masks,
#             then H1b unchanged; the gate uses the R4 count before verification)
# ============================================================

import src.pipeline.hybrid as hybrid
for m in (gpv, hybrid, sg):
    importlib.reload(m)

TD_VOCAB = TD_ROOT / "san_vocab_v1"
TD_TILED = TD_ROOT / "san_vocab_tiled"

# Gate threshold from the validation strata (S2 upper bound)
val_meta = split_meta("val")
val_meta["density_stratum"] = pd.qcut(val_meta["gt_count"].rank(method="first"),
                                      q=[0, .2, .4, .6, .8, .95, 1.0], labels=STRATA).astype(str)
COUNT_GATE = int(val_meta.loc[val_meta["density_stratum"] == "S2_20_40", "gt_count"].max())
print("COUNT_GATE (S2 upper bound):", COUNT_GATE)

def dominant(masks, amax):
    n_cls = int(amax.max()) + 1
    return np.array([np.bincount(amax[m], minlength=n_cls).argmax() if m.any() else -1 for m in masks])

g_rows = []
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="Gated verification"):
    di = r.dataset_index
    stem = io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    b0m, b0meta = io_cache.load_b0_masks(p["b0_masks"])
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    g = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem}.npz")
    keep = gpv.nms_masks(g["masks"], g["scores"], 0.5)
    r4_m, _, _ = gpv.build_r4(g["masks"][keep], g["scores"][keep], b0m, b0meta["predicted_iou"], 0.5)
    d1 = gpv.d1_peaks(san["target_score"])
    with np.load(TD_VOCAB / f"{stem}.npz") as v:
        am = {"full": v["argmax"]}
    with np.load(TD_TILED / f"{stem}.npz") as v:
        am["tiled"] = v["tiled_argmax"]

    n_r4 = int(r4_m.shape[0])
    gate_on = n_r4 <= COUNT_GATE
    row = {"dataset_index": di, "R4_count": n_r4, "gate_on": gate_on}
    for k in ["tiled", "full"]:
        if gate_on and n_r4:
            kv = dominant(r4_m, am[k]) == 0
        else:
            kv = np.ones(n_r4, bool)
        row[f"GPV_R4+gV-a[{k}]"] = int(kv.sum())
        row[f"H1b+gV-a[{k}]"] = float(hybrid.h1b_count(r4_m[kv], d1, a3c, dep, SG_CFG)[0])
    g_rows.append(row)

g_df = pd.DataFrame(g_rows).merge(td_res[["dataset_index", "B0", "GPV_R4_v1", "H1b_v1",
                                          "gt_count", "density_stratum"]], on="dataset_index")
assert (g_df["R4_count"] == g_df["GPV_R4_v1"]).all()
# Gate-off images must equal their ungated parents
off = ~g_df["gate_on"]
assert (g_df.loc[off, "GPV_R4+gV-a[tiled]"] == g_df.loc[off, "GPV_R4_v1"]).all()
assert np.allclose(g_df.loc[off, "H1b+gV-a[tiled]"], g_df.loc[off, "H1b_v1"])
print("Gate ON images by stratum:")
display(g_df.groupby("density_stratum")["gate_on"].agg(["sum", "size", "mean"]).reindex(STRATA).round(2))

COUNT_GATE (S2 upper bound): 20


Gated verification:   0%|          | 0/400 [00:00<?, ?it/s]

Gate ON images by stratum:


,sum,size,mean
density_stratum,,,
S1_0_20,50,80,0.62
S2_20_40,47,80,0.59
S3_40_60,11,60,0.18
S4_60_80,5,60,0.08
S5_80_95,8,60,0.13
S6_95_100,12,60,0.20


In [5]:
# ============================================================
# Step 6.2 — Metrics and the revised targets (T1, T2', T3')
# ============================================================

SYS6 = ["B0", "GPV_R4_v1", "H1b_v1",
        "GPV_R4+gV-a[tiled]", "H1b+gV-a[tiled]", "GPV_R4+gV-a[full]", "H1b+gV-a[full]"]
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
gt6 = g_df["gt_count"].to_numpy(float); st6 = g_df["density_stratum"].to_numpy()
AE6 = {s: np.abs(g_df[s].to_numpy(float) - gt6) for s in SYS6}
IDX6 = {}
for j, k in enumerate(STRATA):
    pos = np.flatnonzero(st6 == k)
    IDX6[k] = pos[np.random.default_rng(1000 + j).integers(0, len(pos), size=(10_000, len(pos)))]

def sci6(a, b, k):
    d = AE6[a][IDX6[k]].mean(1) - AE6[b][IDX6[k]].mean(1)
    return AE6[a][st6 == k].mean() - AE6[b][st6 == k].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

def rci6(a, b):
    d = sum(w * (AE6[a][IDX6[k]].mean(1) - AE6[b][IDX6[k]].mean(1)) for k, w in W.items())
    o = sum(w * (AE6[a][st6 == k].mean() - AE6[b][st6 == k].mean()) for k, w in W.items())
    return o, np.percentile(d, 2.5), np.percentile(d, 97.5)

res6 = pd.DataFrame([{"system": s,
                      "MAE_reweighted": sum(w * AE6[s][st6 == k].mean() for k, w in W.items()),
                      **{f"MAE_{k[:2]}": AE6[s][st6 == k].mean() for k in STRATA}} for s in SYS6])
display(res6.round(2))

t_rows = []
for s in SYS6[3:]:
    rec = {"system": s}
    t1 = []
    for k in ["S1_0_20", "S2_20_40"]:
        d, lo, hi = sci6(s, "B0", k); rec[f"{k[:2]} vs B0"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t1.append(lo <= 0)
    t2 = []
    for k in ["S5_80_95", "S6_95_100"]:
        d, lo, hi = sci6(s, "H1b_v1", k); rec[f"{k[:2]} vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t2.append(lo <= 0)
    d, lo, hi = rci6(s, "H1b_v1"); rec["overall vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
    rec.update({"T1": all(t1), "T2'": all(t2), "T3'": hi < 0, "ALL": all(t1) and all(t2) and hi < 0})
    t_rows.append(rec)
targets6 = pd.DataFrame(t_rows)
print("\nRevised targets (difference [95% CI]; negative = system better):")
display(targets6)

D6 = B3S_DIR / "step6_count_gated_verification"
D6.mkdir(parents=True, exist_ok=True)
g_df.to_csv(D6 / "B3S_s6_per_image.csv", index=False)
res6.to_csv(D6 / "B3S_s6_metrics.csv", index=False)
targets6.to_csv(D6 / "B3S_s6_targets.csv", index=False)
with open(D6 / "B3S_s6_metadata.json", "w") as f:
    json.dump({"step": "6 — count-gated verification (option A, final pre-declared test of this line)",
               "gate": f"apply V-a only if GPV_R4 count <= {COUNT_GATE} (validation S2 upper bound)",
               "V-a": "drop kept mask if dominant expanded-vocabulary class != target",
               "maps": {"primary": "tiled", "secondary": "full"},
               "targets": {"T1": "S1,S2 not significantly worse than B0",
                           "T2'": "S5,S6 not significantly worse than H1b_v1",
                           "T3'": "reweighted MAE significantly better than H1b_v1"},
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", D6)

,system,MAE_reweighted,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0,35.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,10.46,9.49,12.35,14.20,37.33,274.98
3,GPV_R4+gV-a[tiled],30.12,7.91,8.11,12.78,16.40,41.07,298.40
4,H1b+gV-a[tiled],28.66,9.34,9.39,12.52,14.33,38.12,276.50
5,GPV_R4+gV-a[full],30.17,7.82,8.22,12.87,16.50,41.08,298.42
6,H1b+gV-a[full],28.74,9.29,9.60,12.78,14.43,37.92,276.53



Revised targets (difference [95% CI]; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,GPV_R4+gV-a[tiled],"+2.35 [+0.89, +3.91]","+0.38 [-1.15, +1.99]","+3.73 [-0.47, +9.62]","+23.42 [+15.32, +32.55]","+1.47 [+0.49, +2.58]",False,False,False,False
1,H1b+gV-a[tiled],"+3.78 [+1.92, +5.74]","+1.65 [-0.41, +3.96]","+0.78 [+0.20, +1.52]","+1.52 [+0.18, +3.63]","+0.01 [-0.25, +0.26]",False,False,False,False
2,GPV_R4+gV-a[full],"+2.26 [+0.80, +3.81]","+0.49 [-1.05, +2.11]","+3.75 [-0.43, +9.60]","+23.43 [+15.35, +32.60]","+1.52 [+0.54, +2.63]",False,False,False,False
3,H1b+gV-a[full],"+3.72 [+1.86, +5.66]","+1.86 [-0.21, +4.18]","+0.58 [+0.05, +1.28]","+1.55 [+0.17, +3.67]","+0.09 [-0.19, +0.36]",False,False,False,False


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3S_Semantic_Improvement/step6_count_gated_verification


# Step 7 — Evaluate all candidate gate signals at once (diagnostic)

Step 6's count gate failed for three reasons: (1) the R4 count is biased by the errors it must detect (over-counting pushes sparse images out of the gate, merging pulls dense flocks in); (2) the threshold (20, a GT boundary) was applied to an inflated pipeline count; (3) in H1b, V-a can delete rule-A merged masks before H1b uses them.

This step measures, for every train-pool image, the **oracle benefit** of ungated V-a[tiled] (GT, diagnosis only), and how well each GT-free signal predicts it. It also tests **Fix 3** (merged masks exempt from V-a in H1b) and three candidate gates with principled thresholds. **Nothing is adopted here.**

**Commitment:** if no signal or gate is adequate, the verification line closes. If one is, a single gate (plus Fix 3) is pre-declared and tested once against T1, T2′, T3′.

In [4]:
# ============================================================
# Step 7.1 — Per-image signals and the oracle benefit of verification
# ============================================================
#
# Oracle (GT, diagnosis only): for each image, does ungated V-a[tiled]
# move the count closer to GT?   benefit = |base - GT| - |V-a - GT|
#   (> 0 helps, < 0 hurts), for GPV_R4 and H1b.
# GT-free candidate gate signals, computed per image:
#   R4_count, post_count (R4 masks kept by V-a), drop_count, drop_fraction,
#   n_merged (rule-A flags on R4), gate1 (GPV_K Gate 1), sg_gate (SG Stage 2),
#   n_d1_peaks, median_mask_area_frac
# Also: H1b + V-a with rule-A merged masks EXEMPT from verification ("Fix 3").
# ============================================================

from sklearn.metrics import roc_auc_score
import src.pipeline.hybrid as hybrid
for m in (gpv, hybrid, sg):
    importlib.reload(m)
TD_TILED = TD_ROOT / "san_vocab_tiled"

def dominant(masks, amax):
    n_cls = int(amax.max()) + 1
    return np.array([np.bincount(amax[m], minlength=n_cls).argmax() if m.any() else -1 for m in masks])

s5 = pd.read_csv(B3S_DIR / "step5_tiling" / "B3S_s5_per_image.csv").set_index("dataset_index")

sig_rows = []
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="Signals"):
    di = r.dataset_index
    stem = io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    b0m, b0meta = io_cache.load_b0_masks(p["b0_masks"])
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    g = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem}.npz")
    keep = gpv.nms_masks(g["masks"], g["scores"], 0.5)
    r4_m, _, _ = gpv.build_r4(g["masks"][keep], g["scores"][keep], b0m, b0meta["predicted_iou"], 0.5)
    d1 = gpv.d1_peaks(san["target_score"])
    with np.load(TD_TILED / f"{stem}.npz") as v:
        amax = v["tiled_argmax"]
    n = r4_m.shape[0]
    if n:
        is_t = dominant(r4_m, amax) == 0
        merged = gpv.merged_mask_flags(r4_m, gpv.points_per_mask(r4_m, d1), "A")
        areas = r4_m.reshape(n, -1).sum(axis=1)
        keep_fix3 = is_t | merged                                  # merged masks exempt
        h1b_fix3 = float(hybrid.h1b_count(r4_m[keep_fix3], d1, a3c, dep, SG_CFG)[0])
        med_area = float(np.median(areas) / (r4_m.shape[1] * r4_m.shape[2]))
    else:
        is_t = merged = np.zeros(0, bool); h1b_fix3 = 0.0; med_area = np.nan
    sig_rows.append({"dataset_index": di, "R4_count": int(n), "post_count": int(is_t.sum()),
                     "drop_count": int((~is_t).sum()), "drop_fraction": float((~is_t).mean()) if n else 0.0,
                     "n_merged": int(merged.sum()), "n_d1_peaks": int(len(d1)),
                     "median_mask_area_frac": med_area, "H1b+V-a[tiled,fix3]": h1b_fix3})
sig = (pd.DataFrame(sig_rows)
       .merge(td_res[["dataset_index", "gt_count", "density_stratum", "GPV_R4_v1", "H1b_v1", "gate1", "sg_gate"]],
              on="dataset_index")
       .merge(s5[["GPV_R4+V-a[tiled]", "H1b+V-a[tiled]"]], left_on="dataset_index", right_index=True))
assert (sig["R4_count"] == sig["GPV_R4_v1"]).all() and (sig["post_count"] == sig["GPV_R4+V-a[tiled]"]).all()

for base, ver in [("GPV_R4_v1", "GPV_R4+V-a[tiled]"), ("H1b_v1", "H1b+V-a[tiled]")]:
    sig[f"benefit_{base}"] = (sig[base] - sig["gt_count"]).abs() - (sig[ver] - sig["gt_count"]).abs()

print("Oracle: share of images where ungated V-a[tiled] helps / hurts, by stratum")
tab = []
for k in STRATA:
    d = sig[sig["density_stratum"] == k]
    for base in ["GPV_R4_v1", "H1b_v1"]:
        b = d[f"benefit_{base}"]
        tab.append({"stratum": k, "system": base, "helps": (b > 0).mean(), "neutral": (b == 0).mean(),
                    "hurts": (b < 0).mean(), "mean_benefit": b.mean()})
display(pd.DataFrame(tab).round(2))

Signals:   0%|          | 0/400 [00:00<?, ?it/s]

Oracle: share of images where ungated V-a[tiled] helps / hurts, by stratum


,stratum,system,helps,neutral,hurts,mean_benefit
0,S1_0_20,GPV_R4_v1,0.72,0.18,0.10,2.92
1,S1_0_20,H1b_v1,0.76,0.18,0.06,3.74
2,S2_20_40,GPV_R4_v1,0.57,0.25,0.18,1.51
3,S2_20_40,H1b_v1,0.60,0.24,0.16,2.58
4,S3_40_60,GPV_R4_v1,0.45,0.25,0.30,0.35
5,S3_40_60,H1b_v1,0.45,0.27,0.28,0.55
6,S4_60_80,GPV_R4_v1,0.23,0.28,0.48,-2.00
7,S4_60_80,H1b_v1,0.23,0.30,0.47,-2.62
8,S5_80_95,GPV_R4_v1,0.03,0.28,0.68,-6.78
9,S5_80_95,H1b_v1,0.05,0.32,0.63,-7.17


In [5]:
# ============================================================
# Step 7.2 — How well does each GT-free signal predict the oracle benefit?
# ============================================================
#
# AUC for "helps" (benefit > 0) vs "hurts" (benefit < 0); neutral images excluded.
# AUC far from 0.5 (either side) = useful signal for a gate.
# Plus the ceiling: MAE with a PERFECT gate (verify exactly where it helps).
# ============================================================

SIGNALS = ["R4_count", "post_count", "drop_count", "drop_fraction", "n_merged",
           "gate1", "sg_gate", "n_d1_peaks", "median_mask_area_frac"]
rows = []
for base in ["GPV_R4_v1", "H1b_v1"]:
    b = sig[f"benefit_{base}"]
    m = b != 0
    y = (b[m] > 0).astype(int)
    for s in SIGNALS:
        x = sig.loc[m, s].astype(float)
        ok = x.notna()
        auc = roc_auc_score(y[ok], x[ok]) if y[ok].nunique() == 2 else np.nan
        rows.append({"system": base, "signal": s, "n_images": int(ok.sum()), "AUC_helps": auc,
                     "strength": abs(auc - 0.5)})
sig_auc = pd.DataFrame(rows).sort_values(["system", "strength"], ascending=[True, False])
display(sig_auc.round(3))

W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
def rw(pred):
    ae = (pred - sig["gt_count"]).abs()
    return sum(w * ae[sig["density_stratum"] == k].mean() for k, w in W.items())

print("\nCeiling of gating (stratum-reweighted MAE):")
for base, ver in [("GPV_R4_v1", "GPV_R4+V-a[tiled]"), ("H1b_v1", "H1b+V-a[tiled]")]:
    perfect = np.where(sig[f"benefit_{base}"] > 0, sig[ver], sig[base])
    print(f"  {base:<10}: no verification {rw(sig[base]):.2f} | ungated {rw(sig[ver]):.2f} | "
          f"perfect gate {rw(pd.Series(perfect)):.2f}")

,system,signal,n_images,AUC_helps,strength
8,GPV_R4_v1,median_mask_area_frac,298,0.736,0.236
0,GPV_R4_v1,R4_count,298,0.401,0.099
2,GPV_R4_v1,drop_count,298,0.427,0.073
7,GPV_R4_v1,n_d1_peaks,298,0.441,0.059
4,GPV_R4_v1,n_merged,298,0.445,0.055
3,GPV_R4_v1,drop_fraction,298,0.455,0.045
5,GPV_R4_v1,gate1,298,0.462,0.038
6,GPV_R4_v1,sg_gate,298,0.465,0.035
1,GPV_R4_v1,post_count,298,0.465,0.035
17,H1b_v1,median_mask_area_frac,295,0.729,0.229



Ceiling of gating (stratum-reweighted MAE):
  GPV_R4_v1 : no verification 29.94 | ungated 31.33 | perfect gate 28.16
  H1b_v1    : no verification 28.65 | ungated 30.16 | perfect gate 26.53


In [6]:
# ============================================================
# Step 7.3 — Candidate gates with principled thresholds (diagnostic; none adopted)
# ============================================================
#
#   G-b : post-verification count <= 20           (S2 upper bound, on the GT scale)
#   G-c : no rule-A merged mask in the image
#   G-d : G-b AND G-c
#   Fix 3 (H1b): merged masks exempt from V-a, ungated and under each gate
# ============================================================

gates = {"none (H1b_v1)": np.zeros(len(sig), bool),
         "ungated": np.ones(len(sig), bool),
         "G-b post<=20": (sig["post_count"] <= 20).to_numpy(),
         "G-c no merged": (sig["n_merged"] == 0).to_numpy(),
         "G-d both": ((sig["post_count"] <= 20) & (sig["n_merged"] == 0)).to_numpy()}

rows = []
for gname, on in gates.items():
    for label, ver in [("H1b+V-a", "H1b+V-a[tiled]"), ("H1b+V-a fix3", "H1b+V-a[tiled,fix3]"),
                       ("GPV_R4+V-a", "GPV_R4+V-a[tiled]")]:
        base = "GPV_R4_v1" if label.startswith("GPV") else "H1b_v1"
        pred = pd.Series(np.where(on, sig[ver], sig[base]))
        ae = (pred - sig["gt_count"]).abs()
        row = {"gate": gname, "variant": label, "MAE_reweighted": rw(pred)}
        for k in STRATA:
            row[f"MAE_{k[:2]}"] = ae[sig["density_stratum"].to_numpy() == k].mean()
        rows.append(row)
gate_tab = pd.DataFrame(rows)
display(gate_tab.round(2))

print("\nGate firing rate by stratum:")
display(pd.DataFrame({g: pd.Series(on).groupby(sig["density_stratum"].to_numpy()).mean()
                      for g, on in gates.items() if g not in ("none (H1b_v1)", "ungated")}).reindex(STRATA).round(2))
print("Targets on this pool — S1/S2 B0: 5.56 / 7.74 ; S5/S6 H1b: 37.33 / 274.98 ; reweighted H1b: 28.65")

D7 = B3S_DIR / "step7_gate_signals"
D7.mkdir(parents=True, exist_ok=True)
sig.to_csv(D7 / "B3S_s7_signals_per_image.csv", index=False)
sig_auc.to_csv(D7 / "B3S_s7_signal_auc.csv", index=False)
gate_tab.to_csv(D7 / "B3S_s7_candidate_gates.csv", index=False)
with open(D7 / "B3S_s7_metadata.json", "w") as f:
    json.dump({"step": "7 — evaluate candidate gate signals against the oracle benefit (diagnostic)",
               "oracle": "benefit = |base - GT| - |V-a[tiled] - GT|, ungated",
               "fix3": "rule-A merged masks exempt from V-a (H1b)",
               "adopted": None,
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", D7)

,gate,variant,MAE_reweighted,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,none (H1b_v1),H1b+V-a,28.65,10.46,9.49,12.35,14.20,37.33,274.98
1,none (H1b_v1),H1b+V-a fix3,28.65,10.46,9.49,12.35,14.20,37.33,274.98
2,none (H1b_v1),GPV_R4+V-a,29.94,8.62,7.70,12.45,16.13,40.52,297.63
3,ungated,H1b+V-a,30.16,6.72,6.91,11.80,16.82,44.50,300.68
4,ungated,H1b+V-a fix3,29.57,8.31,7.36,11.65,15.52,42.97,291.12
5,ungated,GPV_R4+V-a,31.33,5.70,6.19,12.10,18.13,47.30,316.22
6,G-b post<=20,H1b+V-a,29.49,7.12,7.92,12.93,16.10,41.57,288.78
7,G-b post<=20,H1b+V-a fix3,29.40,8.70,8.21,12.75,15.40,41.12,284.38
8,G-b post<=20,GPV_R4+V-a,30.93,6.09,7.08,13.25,18.08,44.43,307.27
9,G-c no merged,H1b+V-a,28.88,8.89,8.69,12.48,15.98,39.22,275.72



Gate firing rate by stratum:


,G-b post<=20,G-c no merged,G-d both
S1_0_20,0.80,0.61,0.54
S2_20_40,0.74,0.57,0.45
S3_40_60,0.30,0.40,0.17
S4_60_80,0.20,0.53,0.10
S5_80_95,0.18,0.42,0.05
S6_95_100,0.30,0.13,0.02


Targets on this pool — S1/S2 B0: 5.56 / 7.74 ; S5/S6 H1b: 37.33 / 274.98 ; reweighted H1b: 28.65
Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3S_Semantic_Improvement/step7_gate_signals


# Step 8 — Improving GPV's mask set directly

Step 7 closed the image-level gating approach: verification would help (perfect-gate ceiling: H1b 28.65 → 26.53) but no GT-free signal predicts where (best AUC 0.74; counts and merged-mask flags ≈ random). The alternative is to improve the **mask set itself**, so no image-level gate is needed.

**Structural idea:** guided masks already have semantic support (each was prompted at a target peak or region); **B0's automatic masks have none**, yet both enter GPV's pool on equal terms. Verifying **only B0-sourced masks** may remove much of the sparse surplus while sparing the guided masks that cover small objects in dense scenes.

**Step 8.1 (diagnostic):** from Step 5's saved per-mask table, break V-a[tiled] drops down by **source × class × stratum**, and estimate what B0-only verification would do to GPV_R4's count.

In [7]:
# ============================================================
# Step 8.1 — V-a[tiled] drops by source x class x stratum (diagnostic)
# ============================================================
#
# Input: Step 5 per-mask table (GPV_R4 kept masks): source (guided / b0),
#        class (single / multi / empty / redundant; GT diagnosis only),
#        dom_target_tiled (dominant tiled expanded-vocabulary class = target).
# "Useful" = single or multi (real objects); "surplus" = empty or redundant.
# ============================================================

mech = pd.read_csv(B3S_DIR / "step5_tiling" / "B3S_s5_mechanism_masks.csv")
mech["drop"] = ~mech["dom_target_tiled"].astype(bool)
mech["useful"] = mech["class"].isin(["single", "multi"])
n_img = td_df.groupby("density_stratum").size()

# ---- (a) Per image, by stratum and source: masks, drops, and what the drops are
g = mech.groupby(["density_stratum", "source"])
brk = pd.DataFrame({
    "masks": g.size(),
    "dropped": g["drop"].sum(),
    "dropped_surplus": g.apply(lambda d: (d["drop"] & ~d["useful"]).sum()),
    "dropped_useful": g.apply(lambda d: (d["drop"] & d["useful"]).sum()),
})
brk = brk.div(brk.index.get_level_values("density_stratum").map(n_img).values, axis=0)
brk["share_of_drops_that_are_useful"] = brk["dropped_useful"] / brk["dropped"].replace(0, np.nan)
print("(a) Per image: V-a[tiled] drops by source (useful = real objects lost)")
display(brk.reindex(STRATA, level=0).round(2))

# ---- (b) Where do the harmful drops come from? Share of useful-object drops by source
harm = (mech[mech["drop"] & mech["useful"]].groupby(["density_stratum", "source"]).size()
        .unstack(fill_value=0).reindex(STRATA).fillna(0))
harm["b0_share_of_useful_drops"] = harm.get("b0", 0) / harm.sum(axis=1).replace(0, np.nan)
help_ = (mech[mech["drop"] & ~mech["useful"]].groupby(["density_stratum", "source"]).size()
         .unstack(fill_value=0).reindex(STRATA).fillna(0))
help_["b0_share_of_surplus_drops"] = help_.get("b0", 0) / help_.sum(axis=1).replace(0, np.nan)
print("\n(b) Useful-object drops (harm) and surplus drops (benefit), by source:")
display(pd.concat({"harm: useful dropped": harm, "benefit: surplus dropped": help_}, axis=1).round(2))

# ---- (c) Approximate count effect of B0-only verification on GPV_R4
# (Drops B0-sourced masks from the FINAL R4 set. Exact "admission before pooling"
#  would rerun NMS, which could let guided masks overlapping a dropped B0 mask
#  survive; Step 8.2 would compute it exactly.)
per_img = mech.groupby("dataset_index").agg(
    n_all=("drop", "size"),
    n_va=("drop", lambda s: int((~s).sum())),
)
per_img["n_b0only"] = mech.groupby("dataset_index").apply(
    lambda d: int(len(d) - (d["drop"] & (d["source"] == "b0")).sum()))
per_img = per_img.reindex(td_df["dataset_index"], fill_value=0)
eff = td_df.set_index("dataset_index")[["gt_count", "density_stratum"]].join(per_img)
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
rows = []
for col, name in [("n_all", "GPV_R4_v1"), ("n_va", "GPV_R4 + V-a (all masks)"), ("n_b0only", "GPV_R4 + V-a (B0 masks only)")]:
    ae = (eff[col] - eff["gt_count"]).abs()
    row = {"system": name, "MAE_reweighted": sum(w * ae[eff["density_stratum"] == k].mean() for k, w in W.items())}
    for k in STRATA:
        row[f"MAE_{k[:2]}"] = ae[eff["density_stratum"] == k].mean()
    rows.append(row)
print("\n(c) Approximate effect on GPV_R4 (tiled maps):")
display(pd.DataFrame(rows).round(2))
print("Reference — B0: S1 5.56, S2 7.74 | H1b_v1: S5 37.33, S6 274.98, reweighted 28.65")

D8 = B3S_DIR / "step8_b0_only"
D8.mkdir(parents=True, exist_ok=True)
brk.to_csv(D8 / "B3S_s8_drops_by_source.csv")
eff.to_csv(D8 / "B3S_s8_b0only_approx_per_image.csv")
print("Saved to:", D8)

(a) Per image: V-a[tiled] drops by source (useful = real objects lost)


/tmp/ipykernel_1103/2680876331.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  "dropped_surplus": g.apply(lambda d: (d["drop"] & ~d["useful"]).sum()),
/tmp/ipykernel_1103/2680876331.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  "dropped_useful": g.apply(lambda d: (d["drop"] & d["useful"]).sum()),


masks  dropped  dropped_surplus  dropped_useful  \
density_stratum source                                                    
S1_0_20         b0      10.11     2.85             2.75            0.10   
                guided   8.76     1.23             1.19            0.04   
S2_20_40        b0      14.11     3.29             2.92            0.36   
                guided   8.14     1.10             0.90            0.20   
S3_40_60        b0      18.07     2.22             1.93            0.28   
                guided  15.12     1.20             0.87            0.33   
S4_60_80        b0      29.83     5.37             2.97            2.40   
                guided  17.40     3.50             2.40            1.10   
S5_80_95        b0      51.85     5.15             1.85            3.30   
                guided  25.00     2.97             1.98            0.98   
S6_95_100       b0      65.12    10.77             2.15            8.62   
                guided  44.50     7.82             1.53            6.28   

                        share_of_drops_that_are_useful  
density_stratum source                                  
S1_0_20         b0                                0.04  
                guided                            0.03  
S2_20_40        b0                                0.11  
                guided                            0.18  
S3_40_60        b0                                0.13  
                guided                            0.28  
S4_60_80        b0                                0.45  
                guided                            0.31  
S5_80_95        b0                                0.64  
                guided                            0.33  
S6_95_100       b0                                0.80  
                guided                            0.80


(b) Useful-object drops (harm) and surplus drops (benefit), by source:


harm: useful dropped                                  \
source                            b0 guided b0_share_of_useful_drops   
density_stratum                                                        
S1_0_20                            8      3                     0.73   
S2_20_40                          29     16                     0.64   
S3_40_60                          17     20                     0.46   
S4_60_80                         144     66                     0.69   
S5_80_95                         198     59                     0.77   
S6_95_100                        517    377                     0.58   

                benefit: surplus dropped                                   
source                                b0 guided b0_share_of_surplus_drops  
density_stratum                                                            
S1_0_20                              220     95                      0.70  
S2_20_40                             234     72                      0.76  
S3_40_60                             116     52                      0.69  
S4_60_80                             178    144                      0.55  
S5_80_95                             111    119                      0.48  
S6_95_100                            129     92                      0.58


(c) Approximate effect on GPV_R4 (tiled maps):


/tmp/ipykernel_1103/2680876331.py:47: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_img["n_b0only"] = mech.groupby("dataset_index").apply(


,system,MAE_reweighted,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,GPV_R4_v1,29.94,8.62,7.70,12.45,16.13,40.52,297.63
1,GPV_R4 + V-a (all masks),31.33,5.70,6.19,12.10,18.13,47.30,316.22
2,GPV_R4 + V-a (B0 masks only),30.32,6.38,6.01,11.77,16.40,45.27,308.40


Reference — B0: S1 5.56, S2 7.74 | H1b_v1: S5 37.33, S6 274.98, reweighted 28.65
Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3S_Semantic_Improvement/step8_b0_only


# Step 8.2 — V-c diagnostic: does target-pixel share separate background from real objects?

Step 8.1: B0-only verification still loses many real objects in dense scenes (B0 contributes 58–77% of the wrongly dropped objects in S4–S6). The root cause is the verifier's error rate: V-a drops ~12% of real object masks (~20% in S6), whatever their source.

V-a drops a mask whenever the target is not the **dominant** class — so a real object where the target wins on 40% of pixels is dropped like a table mask where it wins on 0%. **V-c (strict)** would drop a mask only if the target wins on **no pixels** (target share = 0; parameter-free).

This diagnostic shows the target-share distributions of surplus vs real-object masks, and what dropping at share = 0 would do. Other cut-offs are shown **only as a sensitivity curve** to judge whether any clean separation exists — not to choose a threshold.

In [8]:
# ============================================================
# Step 8.2 — V-c diagnostic (target-pixel share), from Step 3's saved table
# ============================================================

mf3 = pd.read_csv(B3S_DIR / "step3_expanded_vocabulary" / "B3S_s3_mask_features.csv")
mf3 = mf3[mf3["system"] == "GPV_R4"].copy()
mf3["useful"] = mf3["class"].isin(["single", "multi"])
mf3["region"] = np.where(mf3["density_stratum"].isin(["S1_0_20", "S2_20_40"]), "sparse S1-S2",
                np.where(mf3["density_stratum"].isin(["S4_60_80", "S5_80_95", "S6_95_100"]), "dense S4-S6", "S3"))

# ---- (a) Distribution of target share: surplus vs real objects
q = [0.0, 0.10, 0.25, 0.50, 0.75, 0.90]
dist = (mf3.groupby(["region", "useful"])["target_win_fraction"]
        .describe(percentiles=q[1:])[["count", "mean", "10%", "25%", "50%", "75%"]])
dist["share_exactly_0"] = mf3.groupby(["region", "useful"])["target_win_fraction"].apply(lambda s: (s == 0).mean())
print("(a) Target-pixel share by mask type (useful = real object):")
display(dist.round(3))

# ---- (b) Among masks V-a would drop (dominant != target): target share of each type
va = mf3[~mf3["dominant_is_target"].astype(bool)]
print("\n(b) Masks V-a drops — target share, surplus vs real objects:")
display(va.groupby(["region", "useful"])["target_win_fraction"]
        .describe(percentiles=[0.25, 0.5, 0.75])[["count", "mean", "25%", "50%", "75%"]].round(3))

# ---- (c) Strict rule V-c (share == 0) and a sensitivity curve
gt_s = td_df.set_index("dataset_index")
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}

def evaluate(drop_mask, label):
    kept = mf3[~drop_mask].groupby("dataset_index").size().reindex(gt_s.index, fill_value=0)
    ae = (kept - gt_s["gt_count"]).abs()
    st = gt_s["density_stratum"]
    row = {"rule": label, "MAE_reweighted": sum(w * ae[st == k].mean() for k, w in W.items())}
    for k in STRATA:
        row[f"MAE_{k[:2]}"] = ae[st == k].mean()
    d = mf3[drop_mask]
    for reg in ["sparse S1-S2", "dense S4-S6"]:
        dr = d[d["region"] == reg]
        n_useful = (mf3["useful"] & (mf3["region"] == reg)).sum()
        row[f"{reg}: false-drop rate"] = dr["useful"].sum() / max(n_useful, 1)
        row[f"{reg}: surplus dropped"] = (~dr["useful"]).sum()
    return row

rows = [evaluate(np.zeros(len(mf3), bool), "none (GPV_R4_v1)"),
        evaluate(~mf3["dominant_is_target"].astype(bool), "V-a (dominant != target)"),
        evaluate(mf3["target_win_fraction"] == 0, "V-c strict (share == 0)  [pre-declared candidate]")]
for t in [0.01, 0.05, 0.10, 0.20, 0.30]:
    rows.append(evaluate(mf3["target_win_fraction"] < t, f"sensitivity: share < {t:.2f}"))
vc = pd.DataFrame(rows)
print("\n(c) Effect on GPV_R4 (approximate: masks dropped from the final set):")
display(vc.round(3))
print("Reference — B0: S1 5.56, S2 7.74 | H1b_v1: S5 37.33, S6 274.98 | GPV_R4 reweighted 29.94")

vc.to_csv(D8 / "B3S_s8_vc_diagnostic.csv", index=False)
dist.to_csv(D8 / "B3S_s8_target_share_distribution.csv")
print("Saved to:", D8)

(a) Target-pixel share by mask type (useful = real object):


count   mean    10%    25%    50%  75%  share_exactly_0
region       useful                                                           
S3           False     674.0  0.733  0.000  0.308  1.000  1.0            0.162
             True     1317.0  0.917  0.708  0.985  1.000  1.0            0.026
dense S4-S6  False    1861.0  0.579  0.000  0.000  0.964  1.0            0.297
             True    12161.0  0.870  0.212  0.999  1.000  1.0            0.083
sparse S1-S2 False    1444.0  0.564  0.000  0.000  0.955  1.0            0.306
             True     1846.0  0.926  0.779  0.972  1.000  1.0            0.024


(b) Masks V-a drops — target share, surplus vs real objects:


count   mean  25%    50%    75%
region       useful                                  
S3           False    181.0  0.051  0.0  0.000  0.038
             True      77.0  0.168  0.0  0.103  0.339
dense S4-S6  False    777.0  0.035  0.0  0.000  0.004
             True    1465.0  0.074  0.0  0.000  0.082
sparse S1-S2 False    627.0  0.031  0.0  0.000  0.001
             True      78.0  0.125  0.0  0.000  0.278


(c) Effect on GPV_R4 (approximate: masks dropped from the final set):


,rule,MAE_reweighted,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6,sparse S1-S2: false-drop rate,sparse S1-S2: surplus dropped,dense S4-S6: false-drop rate,dense S4-S6: surplus dropped
0,none (GPV_R4_v1),29.941,8.625,7.700,12.450,16.133,40.517,297.633,0.000,0,0.000,0
1,V-a (dominant != target),32.138,5.638,6.400,12.917,19.917,49.617,314.417,0.042,627,0.120,777
2,V-c strict (share == 0) [pre-declared candidate],31.088,6.438,6.388,12.667,17.833,46.533,308.850,0.024,442,0.083,552
3,sensitivity: share < 0.01,31.146,6.225,6.312,12.733,18.033,46.833,309.200,0.025,504,0.083,608
4,sensitivity: share < 0.05,31.232,6.000,6.238,12.633,18.267,47.400,309.900,0.027,550,0.087,661
5,sensitivity: share < 0.10,31.362,5.988,6.275,12.650,18.483,47.683,310.617,0.028,561,0.092,692
6,sensitivity: share < 0.20,31.496,5.888,6.250,12.533,18.750,48.150,311.783,0.030,584,0.099,721
7,sensitivity: share < 0.30,31.713,5.762,6.262,12.600,19.250,48.667,312.767,0.032,605,0.107,745


Reference — B0: S1 5.56, S2 7.74 | H1b_v1: S5 37.33, S6 274.98 | GPV_R4 reweighted 29.94
Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3S_Semantic_Improvement/step8_b0_only


# B3S conclusion

**Development pool:** train_dev400 (FSC-147 train; 80/80/60/60/60/60) reproduces the held-out pattern that dev100 missed — every semantic addition is significantly worse than B0 in S1 (GPV_R4 +3.06, H1b +4.90).

**Sparse diagnosis (Step 2):** S1 error is surplus masks (GPV over-counts in 74 of 80 images): ~8.8 empty masks per image, half from B0, half from guided prompts.

**Verification line (Steps 3–8) — closed:**
- An expanded vocabulary (43 COCO-Stuff background classes) lets SAN recognise background masks; ungated V-a fixes S1 (GPV_R4 8.62 → 5.64 ≈ B0) and makes S2 better than B0.
- It hurts dense scenes: 8–12% of real-object masks look non-target, and real objects vastly outnumber false masks there (plus the undercount effect).
- About half of the surplus masks look like the target (share ≈ 1): no semantic rule can remove them.
- Not fixed by: size restriction (V-b), SAN tiling (< 1 pp change), count gate (self-defeating), principled gates (≈ 0 gain), B0-only verification (B0 causes 58–77% of dense false drops), strict rule V-c (no separating threshold).
- No GT-free signal predicts where verification helps (best AUC 0.74); perfect-gate ceiling H1b 28.65 → 26.53.

**Next:** B3U — unified per-mask multiplicity (UM) and H1b-K3 (see Fundamental Review).